<div style="background:linear-gradient(120deg,#00553A 0%,#00704A 45%,#00A86A 100%);
            padding:34px 38px 30px 38px;border-radius:6px;color:#fff;
            font-family:Calibri,'Segoe UI',sans-serif">
  <div style="font-size:11px;font-weight:700;letter-spacing:3px;color:#F5C242;text-transform:uppercase">
    03 · NON-TRADITIONAL DATA &nbsp;·&nbsp; LABORATORY
  </div>
  <div style="font-size:38px;font-weight:700;line-height:1.12;margin-top:10px">
    Ookla Speedtest Open Data &amp; WorldPop
  </div>
  <div style="font-size:17px;font-style:italic;color:#E6F6EE;margin-top:8px">
    From crowdsourced measurement tiles to a population-weighted subnational connectivity indicator
  </div>
  <div style="height:5px;background:#F5C242;margin-top:24px;width:120px"></div>
  <div style="font-size:12.5px;color:#E6F6EE;margin-top:18px;line-height:1.6">
    African Development Bank &nbsp;·&nbsp; AU STATAFRIC &nbsp;·&nbsp; STG17 Technical Workshop<br>
    <b>Emerging Issues, Emerging Practice</b> — Innovating the Data Value Chain
  </div>
</div>

## What this notebook does

You give it **one country code**. It gives you back a **published dashboard**.

In between, it runs the full data value chain that an NSO would run in production:

| Step | What happens |
|---|---|
| **Acquire** | Queries the global Ookla Open Data parquet files *remotely* — only the rows covering your country are downloaded (quadkey range pruning, 100–500× less traffic than a full download) |
| **Acquire** | Pulls official administrative boundaries (geoBoundaries) and the WorldPop gridded population raster |
| **Verify** | Runs a **coverage diagnostic** *before* any indicator is designed — how much of the territory and of the population is actually measured |
| **Integrate** | Joins ~600 m measurement tiles to the 1 km population grid and to ADM1/ADM2 boundaries |
| **Analyse** | Population-weighted speeds, coverage gaps, the urban–rural divide, fixed vs mobile, quarterly trends, a connectivity Lorenz curve |
| **Communicate** | Interactive maps, charts and a **single self-contained HTML dashboard** you can publish on GitHub Pages |
| **Document** | Auto-generates the README, the sources block and an explicit **statement of limitations** — part of the deliverable, not an afterthought |

## Learning objectives

By the end of this lab you will be able to:

1. Explain what an Ookla performance tile *is* and what it is **not** (a crowdsourced, self-selected measurement — never a coverage map).
2. Query a multi-hundred-megabyte remote parquet file efficiently with **DuckDB** and quadkey range predicates.
3. Decode a **quadkey** to geographic coordinates yourself, without a helper library.
4. Weight a connectivity statistic by **population** rather than by tile count — and explain why the two answers differ so much.
5. Produce and publish a reproducible analytical product with a documented method, licence and limitations statement.

## How to run this notebook

| Environment | What to do |
|---|---|
| **Google Colab** | `Runtime → Run all`. Missing packages install automatically (~2 min). |
| **Kaggle** | Turn **Internet ON** in the right-hand panel (Settings → Internet), then `Run All`. |
| **Local / JupyterLab** | Python ≥ 3.9. The setup cell installs what is missing. |

**Expected runtime:** 4–9 minutes for a small country (Rwanda, Togo, Djibouti), 10–25 minutes for a large one (Nigeria, DRC, Egypt).
**No API key and no account is required** — every source used here is open.

---

# 1 · Configuration

> **This is the only cell you need to edit.** Everything downstream is derived from it.

Change `COUNTRY_ISO3` to any ISO 3166-1 alpha-3 code and re-run the notebook: `RWA`, `TUN`, `CIV`, `CMR`, `MOZ`, `SOM`, `NGA`, `KEN`, `SEN`, `MAR`, `ZAF`, `EGY`, `GHA`, `TGO`, `DZA`…

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  L A B   C O N F I G U R A T I O N
# ══════════════════════════════════════════════════════════════════════════════

COUNTRY_ISO3 = "RWA"       # ISO-3166-1 alpha-3.  RWA = Rwanda (host country)

# --- Ookla -------------------------------------------------------------------
N_QUARTERS   = 4           # number of most recent quarters to fetch (1 = fastest)
SERVICES     = ["fixed", "mobile"]   # ["mobile"] only is ~2x faster

# --- WorldPop ----------------------------------------------------------------
WORLDPOP_YEAR = 2020       # 2000-2020 available for the global 1 km product
WORLDPOP_RES  = "1km"      # "1km" (recommended, fast) or "100m" (heavy, precise)

# --- Administrative level ----------------------------------------------------
ADMIN_LEVEL   = "ADM2"     # "ADM1" (regions/provinces) or "ADM2" (districts)
                           # falls back to ADM1 automatically if ADM2 is absent

# --- Analytical thresholds ---------------------------------------------------
BROADBAND_MBPS   = 10      # ITU "usable broadband" reference threshold
GOOD_SPEED_MBPS  = 25      # "high-quality connectivity" threshold
MIN_TESTS_TILE   = 1       # drop tiles with fewer than N tests in the quarter
URBAN_DENS_MIN   = 1500    # people / km2 -> urban   (GHSL-inspired proxy)
PERIURBAN_DENS_MIN = 300   # people / km2 -> peri-urban

# --- Output ------------------------------------------------------------------
OUTPUT_DIR    = "outputs"  # dashboard, CSV, GeoJSON and README are written here
CACHE_DIR     = "cache"    # downloaded files are cached here (safe to delete)
MAX_MAP_TILES = 9000       # polygons on the tile map (auto-aggregated above this).
                           # Each language carries its own copy of every map, so this
                           # setting drives the size of the published dashboard.

# ══════════════════════════════════════════════════════════════════════════════
print(f"Country      : {COUNTRY_ISO3}")
print(f"Quarters     : last {N_QUARTERS}")
print(f"Services     : {', '.join(SERVICES)}")
print(f"Admin level  : {ADMIN_LEVEL}")
print(f"WorldPop     : {WORLDPOP_YEAR} @ {WORLDPOP_RES}")

# 2 · Environment

The cell below detects where you are running (Colab / Kaggle / local) and installs
**only what is missing**. It is deliberately written with `subprocess` instead of the
`%pip` magic so that the notebook also runs as a plain script (`jupyter nbconvert --execute`),
which is how it will run in a CI pipeline once you publish it.

In [ ]:
import importlib, subprocess, sys, os, warnings
warnings.filterwarnings("ignore")

# --- where are we? -----------------------------------------------------------
IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
ENV = "Google Colab" if IN_COLAB else ("Kaggle" if IN_KAGGLE else "Local / JupyterLab")

# Kaggle writes only to /kaggle/working
if IN_KAGGLE:
    OUTPUT_DIR = "/kaggle/working/" + OUTPUT_DIR
    CACHE_DIR  = "/kaggle/working/" + CACHE_DIR

REQUIRED = {           # import name  ->  pip name
    "requests":   "requests",
    "pandas":     "pandas",
    "numpy":      "numpy",
    "pyarrow":    "pyarrow",
    "duckdb":     "duckdb",
    "shapely":    "shapely",
    "geopandas":  "geopandas",
    "rasterio":   "rasterio",
    "folium":     "folium",
    "branca":     "branca",
    "plotly":     "plotly",
}

missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
print(f"Environment  : {ENV}")
print(f"Python       : {sys.version.split()[0]}")

if missing:
    print(f"Installing   : {', '.join(missing)} …")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)
    importlib.invalidate_caches()
    print("Installed.")
else:
    print("Dependencies : all present ✓")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR,  exist_ok=True)
print(f"Outputs      : {os.path.abspath(OUTPUT_DIR)}")

In [ ]:
import json, math, time, io, textwrap, datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import duckdb
import geopandas as gpd
import rasterio
from rasterio.warp import transform_bounds
from shapely.geometry import box, Point
import folium
from folium.plugins import HeatMap, Fullscreen, MiniMap
import branca.colormap as cm
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from IPython.display import HTML, display, Markdown

# ══════════════════════════════════════════════════════════════════════════════
#  AfDB-inspired institutional palette (see AfDB_Style_Guide_Presentations.md)
# ══════════════════════════════════════════════════════════════════════════════
GREEN, DEEP, FOREST = "#00A86A", "#00704A", "#00553A"
GOLD,  OCHRE        = "#F5C242", "#D49A00"
TEAL,  TERRA, BRICK = "#0E7C86", "#C4621D", "#B83B2E"
INK,   SLATE        = "#231F20", "#5E6964"
MIST,  MINT,  SAGE  = "#F4F7F5", "#E8F5EF", "#D5DED9"

RAMP    = ["#9ED9C0", "#7BCBA9", "#57BD92", "#33AF7C", "#10A06A", "#008A5B", "#00664A"]
CATCOL  = [GREEN, DEEP, TEAL, OCHRE, TERRA, BRICK]
FONT    = "Calibri, Segoe UI, Helvetica, Arial, sans-serif"

pio.templates["afdb"] = go.layout.Template(layout=dict(
    font=dict(family=FONT, size=13, color=INK),
    colorway=CATCOL,
    paper_bgcolor="white", plot_bgcolor="white",
    title=dict(font=dict(size=18, color=INK)),
    xaxis=dict(gridcolor="#E1E7E4", zeroline=False, linecolor=SAGE,
               tickfont=dict(color=SLATE, size=11)),
    yaxis=dict(gridcolor="#E1E7E4", zeroline=False, linecolor=SAGE,
               tickfont=dict(color=SLATE, size=11)),
    legend=dict(bgcolor="rgba(0,0,0,0)", font=dict(size=11, color=SLATE)),
    margin=dict(l=60, r=30, t=70, b=55),
    hoverlabel=dict(font=dict(family=FONT, size=12), bgcolor="white",
                    bordercolor=SAGE),
))
pio.templates.default = "afdb"


# ══════════════════════════════════════════════════════════════════════════════
#  Small presentation helpers used throughout the notebook
# ══════════════════════════════════════════════════════════════════════════════
def banner(kicker, title, body="", color=GREEN):
    display(HTML(f"""
    <div style="font-family:{FONT};border-left:4px solid {color};background:{MIST};
                padding:12px 16px;margin:6px 0 14px 0;border-radius:0 4px 4px 0">
      <div style="font-size:10.5px;font-weight:700;letter-spacing:2.5px;
                  text-transform:uppercase;color:{DEEP}">{kicker}</div>
      <div style="font-size:15px;font-weight:700;color:{INK};margin-top:3px">{title}</div>
      <div style="font-size:12.5px;color:{SLATE};margin-top:4px;line-height:1.55">{body}</div>
    </div>"""))

def callout(text, kind="info"):
    c = {"info": GREEN, "warn": OCHRE, "risk": BRICK}[kind]
    bg = {"info": MINT, "warn": "#FDF4E0", "risk": "#FBECEA"}[kind]
    icon = {"info": "●", "warn": "▲", "risk": "■"}[kind]
    display(HTML(f"""
    <div style="font-family:{FONT};background:{bg};border:1px solid {c};
                border-radius:5px;padding:11px 15px;margin:10px 0;font-size:12.5px;
                color:{INK};line-height:1.6"><b style="color:{c}">{icon}</b> &nbsp;{text}</div>"""))

def kpi_row(items):
    """items = [(label, value, unit, colour), ...]"""
    cards = "".join(f"""
      <div style="flex:1;min-width:150px;background:white;border:1px solid {SAGE};
                  border-radius:6px;padding:14px 16px">
        <div style="font-size:9.5px;font-weight:700;letter-spacing:2px;
                    text-transform:uppercase;color:{SLATE}">{lab}</div>
        <div style="font-size:30px;font-weight:700;color:{col};line-height:1.15;
                    margin-top:5px">{val}<span style="font-size:13px;font-weight:600;
                    color:{SLATE};margin-left:3px">{unit}</span></div>
      </div>""" for lab, val, unit, col in items)
    display(HTML(f'<div style="display:flex;gap:11px;flex-wrap:wrap;'
                 f'font-family:{FONT};margin:10px 0 16px 0">{cards}</div>'))

def style_table(df, caption=""):
    """Accepts a DataFrame or an already-formatted Styler."""
    sty = df if hasattr(df, "set_table_styles") else df.style
    return (sty
              .set_caption(caption)
              .set_table_styles([
                  {"selector": "caption",
                   "props": [("caption-side", "top"), ("font-family", FONT),
                             ("font-size", "12px"), ("font-style", "italic"),
                             ("color", SLATE), ("padding-bottom", "6px")]},
                  {"selector": "th",
                   "props": [("background-color", DEEP), ("color", "white"),
                             ("font-family", FONT), ("font-size", "11.5px"),
                             ("text-align", "left"), ("padding", "6px 9px")]},
                  {"selector": "td",
                   "props": [("font-family", FONT), ("font-size", "12px"),
                             ("padding", "5px 9px"), ("border-bottom", f"1px solid {SAGE}")]},
              ]))

def fmt(n, d=1):
    """Human-readable number formatting."""
    if n is None or (isinstance(n, float) and not np.isfinite(n)): return "–"
    a = abs(n)
    if a >= 1e9:  return f"{n/1e9:.{d}f} bn"
    if a >= 1e6:  return f"{n/1e6:.{d}f} M"
    if a >= 1e3:  return f"{n/1e3:.{d}f} k"
    return f"{n:.{d}f}"

banner("SETUP COMPLETE", f"Environment ready — {ENV}",
       "AfDB-inspired theme loaded. Every chart, map and dashboard element below "
       "uses the same palette and typography.")

# 3 · Understanding the two data sources

Before writing a single line of analysis, an official statistician needs to be able to answer
three questions about any non-traditional source: **what is the unit of observation, how was it
generated, and what may I legally publish from it?**

## 3.1 Ookla Speedtest Open Data

| Property | Value |
|---|---|
| **Producer** | Ookla® (Speedtest.net) |
| **Unit of observation** | A **tile**: a Web-Mercator zoom-16 square, ≈ **611 m × 611 m at the equator** |
| **Temporal granularity** | **Quarterly**, from **Q1 2019** to the present |
| **Two products** | `fixed` (tests from Wi-Fi / fixed broadband) and `mobile` (tests from a cellular connection) |
| **Key fields** | `quadkey`, `avg_d_kbps`, `avg_u_kbps`, `avg_lat_ms`, `tests`, `devices` |
| **Aggregation rule** | A tile appears only if it received **≥ 1 test** in the quarter; values are averages over all tests in that tile |
| **Licence** | **CC BY-NC-SA 4.0** |
| **Distribution** | Public AWS S3 bucket `ookla-open-data`, GeoParquet + Shapefile |

### What CC BY-NC-SA 4.0 means for an NSO

This is the single most important slide of the session and the one most often skipped.

- **BY** — you must credit Ookla explicitly, in the dashboard and in any publication.
- **NC** — **non-commercial use only**. Selling the derived indicator, or embedding it in a paid product, is not permitted. Publication on an NSO website as a free public good is normally fine; *have your legal service confirm it in writing before the first release*.
- **SA** — **share-alike**: any derivative you distribute must carry the same licence. That includes your dashboard and your aggregated CSV. It also means you **cannot** fold this indicator into an open-data portal that publishes everything under CC BY or CC0 without flagging the exception.
- For an indicator to enter the **official** statistical production system, rather than remaining an experimental release, most offices will need a direct agreement with Ookla or with national operators.

### The bias you must never forget

Ookla data is **crowdsourced and self-selected**. A user runs a test because they *suspect a problem*, or because they *just bought a new connection*. Consequences:

- Tiles with no data are **not** tiles with no coverage — they are tiles with **no test**. Absence of measurement ≠ absence of service.
- Rural and low-income areas are systematically **under-sampled**, so a naive national average is biased **upward**.
- Devices differ (an old phone caps the measured speed), and a tile with 3 tests is not comparable to a tile with 3 000.

This is exactly why we bring in population data.

## 3.2 WorldPop

| Property | Value |
|---|---|
| **Producer** | WorldPop, University of Southampton |
| **Product used** | Global gridded population, UN-adjusted |
| **Resolution** | 1 km (default here) or 100 m |
| **Unit** | Estimated number of **people per grid cell** |
| **Licence** | **CC BY 4.0** — more permissive than Ookla |
| **Method** | Census/projection counts redistributed by a random-forest dasymetric model using built-up area, roads, night lights, land cover |

WorldPop lets us move from *"the average tile in this district records 18 Mbps"* — a statement about **squares** — to *"the median inhabitant of this district lives where 18 Mbps is recorded"* — a statement about **people**. Only the second is a statistic.

> **Caution:** WorldPop is itself a *modelled* product, partly built from night-time lights. Never validate a light-derived indicator against WorldPop and present it as independent confirmation — the circularity is real.

## 3.3 Quadkeys — the geometry hiding in a string

Ookla ships geometry as WKT, but it also ships the `quadkey`, and the quadkey **is** the geometry.
Learning to decode it is worth five minutes because it makes the remote query in §5 possible.

A quadkey is the path down the Web-Mercator quadtree. At each level the world square is divided
into four, numbered:

```
      +---+---+
      | 0 | 1 |
      +---+---+
      | 2 | 3 |
      +---+---+
```

So the string `"0313102310"` means: *take quadrant 0, then 3 of that, then 1 of that…*
Each character adds one zoom level, so a **zoom-16 tile has a 16-character quadkey**.

Three properties matter for us:

1. **Prefix = containment.** Every tile inside the zoom-8 tile `03131023` starts with `03131023`. One string comparison replaces a spatial index.
2. **Lexicographic order ≈ spatial order.** Sorting quadkeys groups neighbours together — which is why a range filter on a sorted parquet file prunes almost everything.
3. **The digits are bit-interleaved (x, y).** Digit `d` contributes bit `d & 1` to `x` and bit `d >> 1` to `y`. That gives us an exact, vectorised decoder in four lines of NumPy.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Quadkey ⇄ geography — no external library, fully vectorised
# ══════════════════════════════════════════════════════════════════════════════
R_EARTH_M = 6378137.0

def quadkeys_to_xy(quadkeys, zoom=16):
    """Vectorised quadkey -> (tile_x, tile_y) integer grid coordinates.

    Each character of the quadkey is a base-4 digit whose bits are interleaved:
    bit 0 of the digit belongs to X, bit 1 belongs to Y.
    """
    qk = np.asarray(quadkeys, dtype=f"U{zoom}")
    # view the unicode array as code points -> shape (n, zoom)
    codes = qk.view(np.uint32).reshape(-1, zoom) - ord("0")
    shifts = (zoom - 1 - np.arange(zoom)).astype(np.uint32)
    x = ((codes & 1) << shifts).sum(axis=1)
    y = ((codes >> 1) << shifts).sum(axis=1)
    return x.astype(np.int64), y.astype(np.int64)


def xy_to_lonlat(x, y, zoom=16):
    """Tile grid coordinates (may be fractional) -> lon/lat in EPSG:4326."""
    n = 2.0 ** zoom
    lon = x / n * 360.0 - 180.0
    lat = np.degrees(np.arctan(np.sinh(np.pi * (1.0 - 2.0 * y / n))))
    return lon, lat


def quadkeys_to_bounds(quadkeys, zoom=16):
    """-> DataFrame with tile centroid and bounding box in degrees."""
    x, y = quadkeys_to_xy(quadkeys, zoom)
    west,  north = xy_to_lonlat(x,     y,     zoom)
    east,  south = xy_to_lonlat(x + 1, y + 1, zoom)
    clon,  clat  = xy_to_lonlat(x + .5, y + .5, zoom)
    return pd.DataFrame({"tile_x": x, "tile_y": y,
                         "west": west, "south": south, "east": east, "north": north,
                         "lon": clon, "lat": clat})


def lonlat_to_quadkey(lon, lat, zoom=16):
    """Scalar lon/lat -> quadkey string. Used to build the remote query."""
    lat = max(min(lat, 85.05112878), -85.05112878)
    n = 2 ** zoom
    x = int((lon + 180.0) / 360.0 * n)
    s = math.sin(math.radians(lat))
    y = int((0.5 - math.log((1 + s) / (1 - s)) / (4 * math.pi)) * n)
    x, y = min(max(x, 0), n - 1), min(max(y, 0), n - 1)
    out = []
    for i in range(zoom, 0, -1):
        digit, mask = 0, 1 << (i - 1)
        if x & mask: digit += 1
        if y & mask: digit += 2
        out.append(str(digit))
    return "".join(out)


def tile_area_km2(lat, zoom=16):
    """True ground area of a Web-Mercator tile at a given latitude."""
    side_m = (2 * np.pi * R_EARTH_M / (2 ** zoom)) * np.cos(np.radians(lat))
    return (side_m ** 2) / 1e6


# --- sanity check: Kigali, Rwanda -------------------------------------------
_qk = lonlat_to_quadkey(30.0619, -1.9441, 16)
_b  = quadkeys_to_bounds([_qk]).iloc[0]
print(f"Kigali (30.0619, -1.9441)")
print(f"  quadkey z16 : {_qk}")
print(f"  tile x / y  : {int(_b.tile_x)} / {int(_b.tile_y)}")
print(f"  centroid    : {_b.lon:.5f}, {_b.lat:.5f}")
print(f"  tile side   : {(2*np.pi*R_EARTH_M/2**16)*np.cos(np.radians(_b.lat)):.0f} m")
print(f"  tile area   : {tile_area_km2(_b.lat):.4f} km²")
assert lonlat_to_quadkey(_b.lon, _b.lat, 16) == _qk, "round-trip failed"
print("  round-trip  : OK ✓")

# 4 · Administrative boundaries

We use **geoBoundaries** (`gbOpen` release, CC BY 4.0, W. M. Geolab / William & Mary) because it
is open, global, versioned and citable. Two caveats to state in your own README:

- geoBoundaries is **not** your national authoritative boundary file. If your NSO or national
  mapping agency publishes official boundaries, use those — replace the single function below and
  everything downstream continues to work.
- Boundary versions change. Record the release you used; the API exposes it.

We also keep the **ADM0** (national) polygon, which is what we will use to clip the Ookla tiles.

In [ ]:
GB_API = "https://www.geoboundaries.org/api/current/gbOpen/{iso}/{lvl}/"
GB_RAW = ("https://raw.githubusercontent.com/wmgeolab/geoBoundaries/main/releaseData/"
          "gbOpen/{iso}/{lvl}/geoBoundaries-{iso}-{lvl}_simplified.geojson")


def get_boundaries(iso3, level):
    """Download a geoBoundaries layer as a GeoDataFrame (API first, raw GitHub fallback)."""
    cache = Path(CACHE_DIR) / f"bnd_{iso3}_{level}.geojson"
    if cache.exists():
        return gpd.read_file(cache)

    url = None
    try:
        meta = requests.get(GB_API.format(iso=iso3, lvl=level), timeout=45).json()
        if isinstance(meta, list):
            meta = meta[0]
        url = meta.get("gjDownloadURL") or meta.get("simplifiedGeometryGeoJSON")
    except Exception:
        pass
    if not url:
        url = GB_RAW.format(iso=iso3, lvl=level)

    gdf = gpd.read_file(url)
    gdf = gdf.set_crs("EPSG:4326", allow_override=True)
    gdf.to_file(cache, driver="GeoJSON")
    return gdf


# --- national outline --------------------------------------------------------
adm0 = get_boundaries(COUNTRY_ISO3, "ADM0")
COUNTRY_NAME = str(adm0.iloc[0].get("shapeGroup", COUNTRY_ISO3))
for c in ("shapeName", "shapeGroupName"):
    if c in adm0.columns and isinstance(adm0.iloc[0][c], str):
        COUNTRY_NAME = adm0.iloc[0][c]
        break
COUNTRY_GEOM = adm0.union_all() if hasattr(adm0, "union_all") else adm0.unary_union
BBOX = COUNTRY_GEOM.bounds                      # (west, south, east, north)

# --- subnational units -------------------------------------------------------
try:
    admin = get_boundaries(COUNTRY_ISO3, ADMIN_LEVEL)
    if admin.empty:
        raise ValueError("empty layer")
except Exception as e:
    print(f"{ADMIN_LEVEL} unavailable ({e}) — falling back to ADM1")
    ADMIN_LEVEL = "ADM1"
    admin = get_boundaries(COUNTRY_ISO3, "ADM1")

admin = admin.rename(columns={"shapeName": "admin_name", "shapeID": "admin_id"})
if "admin_name" not in admin.columns:
    admin["admin_name"] = [f"{ADMIN_LEVEL}-{i+1}" for i in range(len(admin))]
admin["admin_name"] = admin["admin_name"].astype(str)
admin = admin[["admin_name", "geometry"]].reset_index(drop=True)

# administrative names are not guaranteed unique -> disambiguate before any join
dupes = admin["admin_name"].duplicated(keep=False)
if dupes.any():
    admin.loc[dupes, "admin_name"] = (admin.loc[dupes, "admin_name"] + " ("
                                      + (admin.loc[dupes].groupby("admin_name").cumcount() + 1).astype(str)
                                      + ")")
    print(f"{int(dupes.sum())} duplicated {ADMIN_LEVEL} names disambiguated.")
admin["admin_idx"] = admin.index

# a second, coarser level for cross-tabulation when we work at ADM2
if ADMIN_LEVEL == "ADM2":
    try:
        adm1 = get_boundaries(COUNTRY_ISO3, "ADM1").rename(columns={"shapeName": "region"})
        adm1 = adm1[["region", "geometry"]]
    except Exception:
        adm1 = None
else:
    adm1 = None

# --- area in km2 (equal-area projection, never compute area in degrees) ------
admin["area_km2"] = admin.to_crs("EPSG:6933").area / 1e6
COUNTRY_AREA_KM2 = float(gpd.GeoSeries([COUNTRY_GEOM], crs=4326)
                         .to_crs("EPSG:6933").area.iloc[0] / 1e6)

banner("BOUNDARIES LOADED", f"{COUNTRY_NAME} ({COUNTRY_ISO3})",
       f"{len(admin)} {ADMIN_LEVEL} units &nbsp;·&nbsp; "
       f"{COUNTRY_AREA_KM2:,.0f} km² &nbsp;·&nbsp; "
       f"bbox {BBOX[0]:.2f}, {BBOX[1]:.2f} → {BBOX[2]:.2f}, {BBOX[3]:.2f} &nbsp;·&nbsp; "
       f"source: geoBoundaries gbOpen")

display(style_table(admin[["admin_name", "area_km2"]]
                    .sort_values("area_km2", ascending=False).head(8)
                    .style.format({"area_km2": "{:,.0f}"}),
                    f"Largest {ADMIN_LEVEL} units (km²) — first 8 of {len(admin)}"))

# 5 · Acquiring the Ookla tiles — without downloading the planet

## 5.1 Where the data lives

Ookla publishes one parquet file per quarter and per service, on a public S3 bucket:

```
https://ookla-open-data.s3.amazonaws.com/parquet/performance/
    type={fixed|mobile}/year=YYYY/quarter=Q/YYYY-MM-01_performance_{type}_tiles.parquet
```

Each file is **global** and weighs **200 MB to 1 GB**. Rwanda represents roughly **0.02 %** of the rows.
Downloading the whole file to keep 0.02 % of it is the single most common mistake in this lab.

## 5.2 The technique: HTTP range requests + quadkey range pruning

Parquet is a **columnar** format organised in *row groups*, and each row group stores the
**min and max value of every column** in its footer. A query engine that can read the footer over
HTTP can therefore:

1. read the footer (a few kilobytes),
2. discard every row group whose `[min(quadkey), max(quadkey)]` interval does not intersect our
   country's quadkey ranges,
3. fetch, with HTTP range requests, **only the surviving byte ranges**, and
4. decompress **only the columns we asked for**.

Because the files are sorted by quadkey and quadkeys are spatially coherent (§3.3), a country
usually survives in a handful of row groups. In practice: **tens of megabytes transferred instead of
hundreds**, in a few seconds.

We use **DuckDB** with the `httpfs` extension for this, and fall back to `pyarrow.dataset` if the
extension cannot be installed (some locked-down corporate environments block it).

## 5.3 Building the predicate

We cover the country bounding box with tiles at a **low zoom** (typically z = 6–9), convert each to
its quadkey prefix, and turn every prefix `p` into a closed string interval:

```
p + "0000…"   ≤  quadkey  ≤   p + "3333…"
```

We also add one **global** `BETWEEN min … max` clause: a single range predicate is the form the
engine pushes down to the row-group statistics most reliably. The per-prefix clauses then remove
the tiles that fall in the bounding box but not in the country.

In [ ]:
OOKLA_BASE = "https://ookla-open-data.s3.amazonaws.com/parquet/performance"
Q_MONTH = {1: "01", 2: "04", 3: "07", 4: "10"}


def ookla_url(service, year, quarter):
    return (f"{OOKLA_BASE}/type={service}/year={year}/quarter={quarter}/"
            f"{year}-{Q_MONTH[quarter]}-01_performance_{service}_tiles.parquet")


def quarter_exists(service, year, quarter, timeout=25):
    try:
        r = requests.head(ookla_url(service, year, quarter), timeout=timeout)
        return r.status_code == 200
    except Exception:
        return False


def previous_quarter(year, quarter):
    return (year - 1, 4) if quarter == 1 else (year, quarter - 1)


def recent_quarters(n, services=("mobile",), max_back=10):
    """Return the n most recent quarters that are actually published, newest first."""
    today = dt.date.today()
    y, q = today.year, (today.month - 1) // 3 + 1
    found, tried = [], 0
    while len(found) < n and tried < max_back:
        if all(quarter_exists(s, y, q) for s in services):
            found.append((y, q))
        elif not found:
            pass                      # still walking back to the latest published quarter
        y, q = previous_quarter(y, q)
        tried += 1
    return found


QUARTERS = recent_quarters(N_QUARTERS, services=SERVICES)
if not QUARTERS:
    raise RuntimeError("No published Ookla quarter found — check your internet connection.")

LATEST_Y, LATEST_Q = QUARTERS[0]
banner("OOKLA CATALOGUE", f"Latest published quarter: {LATEST_Y} Q{LATEST_Q}",
       "Quarters to be fetched: " +
       " · ".join(f"{y} Q{q}" for y, q in QUARTERS) +
       f"<br>Example file: <code style='font-size:11px'>{ookla_url(SERVICES[0], LATEST_Y, LATEST_Q)}</code>")

In [ ]:
def bbox_quadkey_prefixes(bbox, max_prefixes=160):
    """Cover a lon/lat bbox with quadkey prefixes, choosing the deepest zoom that
    keeps the number of prefixes manageable (deeper = more selective pruning)."""
    west, south, east, north = bbox
    for zoom in range(10, 2, -1):
        n = 2 ** zoom
        x0 = int((west + 180) / 360 * n); x1 = int((east + 180) / 360 * n)
        def _y(lat):
            s = math.sin(math.radians(max(min(lat, 85.05), -85.05)))
            return int((0.5 - math.log((1 + s) / (1 - s)) / (4 * math.pi)) * n)
        y0, y1 = _y(north), _y(south)
        count = (x1 - x0 + 1) * (y1 - y0 + 1)
        if count <= max_prefixes:
            prefixes = []
            for xx in range(x0, x1 + 1):
                for yy in range(y0, y1 + 1):
                    lon = (xx + 0.5) / n * 360 - 180
                    lat = math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * (yy + 0.5) / n))))
                    prefixes.append(lonlat_to_quadkey(lon, lat, zoom))
            return sorted(set(prefixes)), zoom
    raise RuntimeError("Could not build quadkey prefixes for this bounding box.")


PREFIXES, PREFIX_ZOOM = bbox_quadkey_prefixes(BBOX)
PAD = 16 - PREFIX_ZOOM
RANGES = [(p + "0" * PAD, p + "3" * PAD) for p in PREFIXES]
GLOBAL_LO, GLOBAL_HI = RANGES[0][0], RANGES[-1][1]

print(f"Bounding box       : {BBOX[0]:.3f}, {BBOX[1]:.3f} → {BBOX[2]:.3f}, {BBOX[3]:.3f}")
print(f"Prefix zoom level  : z{PREFIX_ZOOM}  ({len(PREFIXES)} prefixes)")
print(f"Global range       : {GLOBAL_LO}  →  {GLOBAL_HI}")
print(f"First prefixes     : {', '.join(PREFIXES[:6])}{' …' if len(PREFIXES) > 6 else ''}")
print(f"\nSelectivity        : the global range covers "
      f"{100 * (len(PREFIXES) / 4 ** PREFIX_ZOOM):.4f}% of the world's z{PREFIX_ZOOM} tiles")

In [ ]:
OOKLA_COLS = ["quadkey", "avg_d_kbps", "avg_u_kbps", "avg_lat_ms", "tests", "devices"]

_DUCK = None
def duck():
    """Lazily create a DuckDB connection with httpfs enabled."""
    global _DUCK
    if _DUCK is None:
        con = duckdb.connect()
        try:
            con.execute("INSTALL httpfs; LOAD httpfs;")
        except Exception as e:
            print(f"  httpfs unavailable ({e}) — will use the pyarrow fallback")
            raise
        con.execute("SET enable_progress_bar=false;")
        _DUCK = con
    return _DUCK


def available_columns(url):
    """Read only the parquet footer to see which columns this release actually has."""
    d = duck().execute(f"DESCRIBE SELECT * FROM read_parquet('{url}') LIMIT 0").df()
    return list(d["column_name"])


def _fetch_duckdb(url):
    cols = [c for c in OOKLA_COLS if c in available_columns(url)] or OOKLA_COLS
    where = " OR ".join(f"(quadkey >= '{lo}' AND quadkey <= '{hi}')" for lo, hi in RANGES)
    sql = f"""
        SELECT {', '.join(cols)}
        FROM read_parquet('{url}')
        WHERE quadkey >= '{GLOBAL_LO}' AND quadkey <= '{GLOBAL_HI}'
          AND ({where})
    """
    return duck().execute(sql).df()


def _fetch_pyarrow(url):
    """Fallback: pyarrow dataset over HTTP with the same range predicate."""
    import pyarrow.dataset as ds, pyarrow.compute as pc, fsspec
    fs, path = fsspec.core.url_to_fs(url)
    dataset = ds.dataset(path, filesystem=fs, format="parquet")
    f = None
    for lo, hi in RANGES:
        cond = (pc.field("quadkey") >= lo) & (pc.field("quadkey") <= hi)
        f = cond if f is None else (f | cond)
    return dataset.to_table(columns=OOKLA_COLS, filter=f).to_pandas()


def fetch_ookla(service, year, quarter, verbose=True):
    """Country subset of one Ookla quarter — cached locally as parquet."""
    cache = Path(CACHE_DIR) / f"ookla_{COUNTRY_ISO3}_{service}_{year}Q{quarter}.parquet"
    if cache.exists():
        df = pd.read_parquet(cache)
        if verbose:
            print(f"  {service:<6} {year} Q{quarter}  ·  {len(df):>7,} tiles  (cache)")
        return df

    url = ookla_url(service, year, quarter)
    t0 = time.time()
    try:
        df = _fetch_duckdb(url)
    except Exception as e:
        if verbose:
            print(f"  DuckDB path failed ({type(e).__name__}) — trying pyarrow …")
        try:
            df = _fetch_pyarrow(url)
        except Exception as e2:
            raise RuntimeError(
                f"Could not read {url}\nDuckDB: {e}\npyarrow: {e2}\n"
                "On Kaggle, check that Internet is switched ON in the settings panel."
            ) from e2

    df["service"], df["year"], df["quarter"] = service, year, quarter
    df.to_parquet(cache, index=False)
    if verbose:
        print(f"  {service:<6} {year} Q{quarter}  ·  {len(df):>7,} tiles  "
              f"({time.time() - t0:5.1f}s remote)")
    return df


print(f"Fetching {len(SERVICES)} service(s) × {len(QUARTERS)} quarter(s) "
      f"from the global parquet files …\n")
frames = []
for (y, q) in QUARTERS:
    for s in SERVICES:
        try:
            frames.append(fetch_ookla(s, y, q))
        except Exception as e:
            print(f"  ! {s} {y}Q{q} skipped: {e}")

raw = pd.concat(frames, ignore_index=True)
print(f"\nTotal rows retrieved (bounding box): {len(raw):,}")

# 6 · Cleaning, geocoding and clipping

Three operations, in this order:

1. **Units.** Ookla stores speeds in **kbps**; every published indicator should be in **Mbps**
   (`Mbps = kbps / 1000`). Getting this wrong by a factor of 1000 is the classic first-run error.
2. **Geometry.** We decode the quadkey with the vectorised function from §3.3 — no WKT parsing,
   which would be an order of magnitude slower on large countries.
3. **Clip.** The bounding box is a rectangle; the country is not. We keep the tiles whose
   **centroid** falls inside the ADM0 polygon. Centroid-in-polygon is the standard, reproducible
   convention for a 611 m grid; state it in your methodology.

In [ ]:
df = raw.copy()

# 1 · units and derived fields ------------------------------------------------
df["d_mbps"] = df["avg_d_kbps"] / 1000.0
df["u_mbps"] = df["avg_u_kbps"] / 1000.0
df["latency_ms"] = df["avg_lat_ms"] if "avg_lat_ms" in df.columns else np.nan
df["tests_per_device"] = df["tests"] / df["devices"].replace(0, np.nan)

# 2 · geometry from the quadkey ----------------------------------------------
bounds = quadkeys_to_bounds(df["quadkey"].values, zoom=16)
df = pd.concat([df.reset_index(drop=True), bounds], axis=1)
df["tile_km2"] = tile_area_km2(df["lat"].values)

# 3 · clip to the national polygon (centroid rule) ---------------------------
pts = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs="EPSG:4326")
inside = pts.within(COUNTRY_GEOM)
n_before = len(df)
df = df[inside.values].reset_index(drop=True)

# 4 · quality filter ----------------------------------------------------------
df = df[df["tests"] >= MIN_TESTS_TILE]
df = df[(df["d_mbps"] > 0) & (df["d_mbps"] < 5000)].reset_index(drop=True)

banner("TILES CLEANED", f"{len(df):,} tiles retained inside {COUNTRY_NAME}",
       f"{n_before - len(df):,} rows dropped: outside the national polygon, "
       f"below {MIN_TESTS_TILE} test(s), or out of physical range. "
       f"Speeds converted kbps → Mbps.")

display(style_table(
    df[["quadkey", "service", "year", "quarter", "lon", "lat",
        "d_mbps", "u_mbps", "latency_ms", "tests", "devices"]].head(6)
      .style.format({"lon": "{:.4f}", "lat": "{:.4f}", "d_mbps": "{:.1f}",
                     "u_mbps": "{:.1f}", "latency_ms": "{:.0f}"}),
    "Cleaned Ookla tiles — first 6 rows"))

# 7 · Coverage diagnostic — *before* designing any indicator

> **Verify national coverage before designing any indicator.**

This is the step that separates an experimental statistic from a misleading one. The question is
not "what is the average speed?" but **"on how much of the country, and on how many people, do I
actually have a measurement?"**

We answer it on three axes:

- **Spatial**: share of the national land area covered by at least one measured tile.
- **Volume**: how many tests, and how concentrated they are (a few tiles can carry most of the tests).
- **Stability**: is the number of measured tiles stable across quarters, or collapsing?

If spatial coverage is below a few percent, a *national* average is not defensible — but a
*subnational, population-weighted, urban-only* indicator often still is. Say which one you are
publishing.

In [ ]:
latest = df[(df.year == LATEST_Y) & (df.quarter == LATEST_Q)]

diag = []
for s in SERVICES:
    d = latest[latest.service == s]
    if d.empty:
        continue
    covered_km2 = d["tile_km2"].sum()
    # test concentration: share of tests carried by the busiest 1% of tiles
    t = np.sort(d["tests"].values)[::-1]
    top1 = t[:max(1, len(t) // 100)].sum() / t.sum() * 100 if t.sum() else np.nan
    diag.append({
        "Service": s,
        "Tiles measured": len(d),
        "Area covered (km²)": covered_km2,
        "Land area covered (%)": 100 * covered_km2 / COUNTRY_AREA_KM2,
        "Tests": int(d["tests"].sum()),
        "Devices": int(d["devices"].sum()),
        "Tests in busiest 1% of tiles (%)": top1,
        "Median tests / tile": float(d["tests"].median()),
    })
diag = pd.DataFrame(diag)

display(style_table(diag.style.format({
    "Tiles measured": "{:,.0f}", "Area covered (km²)": "{:,.0f}",
    "Land area covered (%)": "{:.2f}", "Tests": "{:,.0f}", "Devices": "{:,.0f}",
    "Tests in busiest 1% of tiles (%)": "{:.1f}", "Median tests / tile": "{:.0f}"}),
    f"Coverage diagnostic — {COUNTRY_NAME}, {LATEST_Y} Q{LATEST_Q}"))

_land = diag["Land area covered (%)"].max() if not diag.empty else 0
if _land < 2:
    callout(f"<b>Spatial coverage is very low ({_land:.2f}% of the land area).</b> "
            "A national average would be a statement about a handful of urban squares. "
            "Publish an urban/population-weighted indicator with an explicit denominator, "
            "or treat the output as a diagnostic tool only.", "risk")
elif _land < 10:
    callout(f"<b>Spatial coverage is {_land:.2f}% of the land area</b> — normal for this source. "
            "Population weighting (§9) is mandatory before any headline figure is quoted.", "warn")
else:
    callout(f"Spatial coverage reaches {_land:.2f}% of the land area — comparatively high. "
            "Population weighting is still required, but subnational breakdowns will be robust.", "info")

In [ ]:
# --- stability across quarters ----------------------------------------------
if len(QUARTERS) > 1:
    stab = (df.groupby(["year", "quarter", "service"])
              .agg(tiles=("quadkey", "size"), tests=("tests", "sum"),
                   median_d=("d_mbps", "median"))
              .reset_index())
    stab["period"] = stab["year"].astype(str) + " Q" + stab["quarter"].astype(str)
    stab = stab.sort_values(["year", "quarter"])
    display(style_table(
        stab.pivot(index="period", columns="service", values="tiles")
            .fillna(0).astype(int).reset_index()
            .style.format(thousands=","),
        "Number of measured tiles per quarter — a sudden drop usually means a data issue, not a network change"))
else:
    stab = None
    print("Single quarter selected — set N_QUARTERS > 1 to enable the stability check.")

# 8 · WorldPop — bringing people into the picture

We download the **UN-adjusted global gridded population** raster for the country. Two products are
wired in:

| Setting | Product | Typical size | Use |
|---|---|---|---|
| `"1km"` | `Global_2000_2020_1km_UNadj` | 0.1 – 20 MB | **Default.** Fast, sufficient at ADM1/ADM2 |
| `"100m"` | `Global_2000_2020` (unconstrained) | 20 – 600 MB | Tile-level precision, slow for large countries |

The UN-adjusted version rescales the grid so that the **national total matches the UN World
Population Prospects** estimate — which is what makes the result comparable with the rest of your
statistical system.

In [ ]:
WP_1KM  = ("https://data.worldpop.org/GIS/Population/Global_2000_2020_1km_UNadj/"
           "{year}/{ISO}/{iso}_ppp_{year}_1km_Aggregated_UNadj.tif")
WP_100M = ("https://data.worldpop.org/GIS/Population/Global_2000_2020/"
           "{year}/{ISO}/{iso}_ppp_{year}.tif")
WP_REST = "https://www.worldpop.org/rest/data/pop/wpgp?iso3={ISO}"


def download(url, dest, label=""):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 1024:
        print(f"  {label or dest.name}: cached ({dest.stat().st_size/1e6:.1f} MB)")
        return dest
    with requests.get(url, stream=True, timeout=180) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        done, t0 = 0, time.time()
        with open(dest, "wb") as fh:
            for chunk in r.iter_content(1 << 20):
                fh.write(chunk); done += len(chunk)
                if total:
                    pct = 100 * done / total
                    print(f"\r  {label} {pct:5.1f}%  ({done/1e6:.1f}/{total/1e6:.1f} MB)",
                          end="", flush=True)
    print(f"\r  {label}: {done/1e6:.1f} MB in {time.time()-t0:.1f}s" + " " * 20)
    return dest


def worldpop_raster(iso3, year, res):
    tpl = WP_1KM if res == "1km" else WP_100M
    url = tpl.format(year=year, ISO=iso3.upper(), iso=iso3.lower())
    dest = Path(CACHE_DIR) / f"worldpop_{iso3}_{year}_{res}.tif"
    try:
        return download(url, dest, f"WorldPop {iso3} {year} {res}")
    except Exception as e:
        print(f"  direct URL failed ({e}) — querying the WorldPop REST catalogue …")
        meta = requests.get(WP_REST.format(ISO=iso3.upper()), timeout=60).json()
        files = [f for rec in meta.get("data", []) if str(rec.get("popyear")) == str(year)
                 for f in rec.get("files", []) if str(f).endswith(".tif")]
        if not files:
            raise RuntimeError(f"No WorldPop raster found for {iso3} {year}")
        return download(files[0], dest, f"WorldPop {iso3} (REST)")


POP_TIF = worldpop_raster(COUNTRY_ISO3, WORLDPOP_YEAR, WORLDPOP_RES)

with rasterio.open(POP_TIF) as src:
    print(f"\n  CRS         : {src.crs}")
    print(f"  Size        : {src.width} × {src.height} px")
    print(f"  Pixel size  : {abs(src.transform.a):.6f}° ≈ "
          f"{abs(src.transform.a) * 111.32:.2f} km at the equator")
    print(f"  NoData      : {src.nodata}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Raster  ->  tidy table of populated cells
# ══════════════════════════════════════════════════════════════════════════════
with rasterio.open(POP_TIF) as src:
    win = rasterio.windows.from_bounds(*BBOX, transform=src.transform)
    win = win.round_offsets().round_lengths()
    win = rasterio.windows.intersection(
        win, rasterio.windows.Window(0, 0, src.width, src.height))
    arr = src.read(1, window=win).astype("float64")
    wtr = src.window_transform(win)
    nod = src.nodata

valid = np.isfinite(arr) & (arr > -1e30)
if nod is not None:
    valid &= (arr != nod)
arr = np.where(valid, np.clip(arr, 0, None), np.nan)

rows, cols = np.nonzero(valid)
pop_vals = arr[rows, cols]

# cell centroids (pixel centre) in EPSG:4326
clon = wtr.c + wtr.a * (cols + 0.5) + wtr.b * (rows + 0.5)
clat = wtr.f + wtr.d * (cols + 0.5) + wtr.e * (rows + 0.5)

# true ground area of each cell
dlon, dlat = abs(wtr.a), abs(wtr.e)
cell_km2 = (dlat * 111.32) * (dlon * 111.32 * np.cos(np.radians(clat)))

cells = pd.DataFrame({
    "row": rows.astype(np.int64), "col": cols.astype(np.int64),
    "lon": clon, "lat": clat, "pop": pop_vals, "cell_km2": cell_km2,
})
cells["cell_id"] = cells["row"] * (arr.shape[1] + 1) + cells["col"]
cells["density"] = cells["pop"] / cells["cell_km2"]

# clip to the national polygon
cpts = gpd.GeoDataFrame(cells, geometry=gpd.points_from_xy(cells.lon, cells.lat), crs="EPSG:4326")
cells = cells[cpts.within(COUNTRY_GEOM).values].reset_index(drop=True)

POP_TOTAL = float(cells["pop"].sum())
kpi_row([
    ("Populated cells", f"{len(cells):,}", "", DEEP),
    ("Total population", fmt(POP_TOTAL), f"({WORLDPOP_YEAR})", GREEN),
    ("Mean density", f"{POP_TOTAL / COUNTRY_AREA_KM2:,.0f}", "/km²", TEAL),
    ("Grid resolution", f"{dlon * 111.32:.2f}", "km", OCHRE),
])
callout("Cross-check this total against your own national projection for "
        f"{WORLDPOP_YEAR}. A gap of more than a few percent is worth a sentence in your "
        "methodology — WorldPop is a model, not a census.", "warn")

# 9 · Spatial integration — the heart of the lab

We now have two grids that do **not** align:

```
   WorldPop  :  ~1 000 m cells,  complete coverage of the territory
   Ookla     :  ~611 m tiles,    only where somebody ran a test
```

Rather than pretending to a precision we do not have, we use an explicit and defensible rule:

1. **Locate every Ookla tile in the population grid** by its centroid → each tile belongs to exactly
   one WorldPop cell.
2. A cell is **measured** if it contains at least one tile; otherwise it is a **measurement gap**.
   This gives us the denominator that matters: `population living in a measured cell / total population`.
3. **Allocate** the population of a measured cell equally among the tiles it contains. Each tile
   receives a population weight `pop_cell / n_tiles_in_cell`. Summed over the country, the allocated
   population equals exactly the population of the measured cells — the weights are consistent.

> **Why not a proper areal-weighted intersection?** Because with a 1 km population grid it would add
> false precision, not accuracy. If you switch `WORLDPOP_RES` to `"100m"`, the reverse assignment
> (population cell → containing tile) becomes exact and this code path handles it unchanged.

The operation below is pure NumPy — no spatial join — and runs in milliseconds even for Nigeria.

In [ ]:
# --- 1 · every tile -> its population cell -----------------------------------
inv = ~wtr                                        # inverse affine transform
tcol, trow = inv * (df["lon"].values, df["lat"].values)
r = np.floor(trow).astype(np.int64)
c = np.floor(tcol).astype(np.int64)
ok = (r >= 0) & (r < arr.shape[0]) & (c >= 0) & (c < arr.shape[1])
df["row"], df["col"] = r, c
df["cell_id"] = np.where(ok, r * (arr.shape[1] + 1) + c, -1)

# --- 2 · how many tiles share each cell (per service and quarter) ------------
key = ["cell_id", "service", "year", "quarter"]
df["n_tiles_in_cell"] = df.groupby(key)["quadkey"].transform("size")

# --- 3 · attach cell population and allocate it ------------------------------
df = df.merge(cells[["cell_id", "pop", "cell_km2", "density"]], on="cell_id", how="left")
df["pop"] = df["pop"].fillna(0.0)
df["pop_tile"] = df["pop"] / df["n_tiles_in_cell"]

# --- 4 · urbanisation class (density proxy, GHSL-inspired) -------------------
def urban_class(d):
    return np.where(d >= URBAN_DENS_MIN, "Urban",
           np.where(d >= PERIURBAN_DENS_MIN, "Peri-urban", "Rural"))

df["settlement"]    = urban_class(df["density"].fillna(0).values)
cells["settlement"] = urban_class(cells["density"].values)

# --- 5 · measured / gap flag, per service, latest quarter --------------------
for s in SERVICES:
    measured = set(df.loc[(df.service == s) & (df.year == LATEST_Y) &
                          (df.quarter == LATEST_Q), "cell_id"])
    cells[f"measured_{s}"] = cells["cell_id"].isin(measured)
cells["measured_any"] = cells[[f"measured_{s}" for s in SERVICES]].any(axis=1)

alloc = df.loc[(df.year == LATEST_Y) & (df.quarter == LATEST_Q)].groupby("service")["pop_tile"].sum()
print("Population allocated to measured tiles (latest quarter)")
for s in SERVICES:
    cov = cells.loc[cells[f"measured_{s}"], "pop"].sum()
    print(f"  {s:<6}: {fmt(alloc.get(s, 0)):>8} allocated  ·  "
          f"{fmt(cov):>8} living in a measured cell  ·  "
          f"{100*cov/POP_TOTAL:5.1f}% of the population")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Attach administrative units (one spatial join for cells, one for tiles)
# ══════════════════════════════════════════════════════════════════════════════
def attach_admin(frame, admin_gdf, label="admin_name"):
    g = gpd.GeoDataFrame(frame[["lon", "lat"]].copy(),
                         geometry=gpd.points_from_xy(frame["lon"], frame["lat"]),
                         crs="EPSG:4326")
    j = gpd.sjoin(g, admin_gdf[[label, "geometry"]], how="left", predicate="within")
    j = j[~j.index.duplicated(keep="first")]
    return j[label].reindex(frame.index).values

t0 = time.time()
cells["admin_name"] = attach_admin(cells, admin)
df["admin_name"]    = attach_admin(df, admin)
if adm1 is not None:
    cells["region"] = attach_admin(cells, adm1, "region")
    df["region"]    = attach_admin(df, adm1, "region")
print(f"Spatial joins done in {time.time()-t0:.1f}s "
      f"({len(cells):,} cells + {len(df):,} tiles)")

unmatched = df["admin_name"].isna().mean() * 100
if unmatched > 1:
    callout(f"{unmatched:.1f}% of tiles fall outside every {ADMIN_LEVEL} polygon "
            "(usually coastline or lake generalisation). They are kept in national totals "
            "but excluded from subnational tables.", "warn")

# 10 · Building the indicators

Four definitions, which you should copy verbatim into your metadata.

**1 · Population coverage of measurement**
$$\text{cov}_{pop} = \frac{\sum_{c \in \mathcal{M}} P_c}{\sum_{c} P_c}$$
where $\mathcal{M}$ is the set of population cells containing at least one measured tile.
*It measures where Speedtest users are, not where the network is.*

**2 · Population-weighted mean download speed**
$$\bar{S}_{pop} = \frac{\sum_t w_t S_t}{\sum_t w_t}, \qquad w_t = \frac{P_{c(t)}}{n_{c(t)}}$$
The tile-count-weighted mean (the naive one) gives every 611 m square the same voice, whether it
holds 4 people or 4 000. Compare the two: the gap is your headline.

**3 · Population-weighted median** — same weights, but the 50th percentile of the weighted
distribution. **Report the median, not the mean**: speed distributions are strongly right-skewed and
a handful of fibre tiles will drag any mean upward.

**4 · Share of the measured population above a service threshold**
$$\pi_{\ge x} = \frac{\sum_{t : S_t \ge x} w_t}{\sum_t w_t}$$
computed at 10 Mbps (ITU usable-broadband reference) and 25 Mbps.

In [ ]:
IND_COLS = ["pop_tile", "d_mbps", "u_mbps", "latency_ms", "tests", "devices", "tile_km2"]


def wq(values, weights, q):
    """Weighted quantile (linear interpolation on the cumulative weight)."""
    v = np.asarray(values, float); w = np.asarray(weights, float)
    m = np.isfinite(v) & np.isfinite(w) & (w > 0)
    if m.sum() == 0:
        return np.nan
    v, w = v[m], w[m]
    o = np.argsort(v); v, w = v[o], w[o]
    cw = np.cumsum(w) - 0.5 * w
    return float(np.interp(q * w.sum(), cw, v))


def indicators(g):
    """Full indicator set for one group of tiles (one service, one area, one quarter)."""
    w, s = g["pop_tile"].values, g["d_mbps"].values
    W = np.nansum(w)
    out = {
        "tiles":        len(g),
        "tests":        float(g["tests"].sum()),
        "devices":      float(g["devices"].sum()),
        "pop_weight":   float(W),
        "area_km2":     float(g["tile_km2"].sum()),
        "d_mean_tile":  float(np.nanmean(s)),
        "d_median_tile": float(np.nanmedian(s)),
        "d_mean_pop":   float(np.nansum(w * s) / W) if W > 0 else np.nan,
        "d_median_pop": wq(s, w, 0.50),
        "d_p10_pop":    wq(s, w, 0.10),
        "d_p90_pop":    wq(s, w, 0.90),
        "u_median_pop": wq(g["u_mbps"].values, w, 0.50),
        "lat_median_pop": wq(g["latency_ms"].values, w, 0.50),
        "pct_above_bb": float(np.nansum(w[s >= BROADBAND_MBPS]) / W * 100) if W > 0 else np.nan,
        "pct_above_hi": float(np.nansum(w[s >= GOOD_SPEED_MBPS]) / W * 100) if W > 0 else np.nan,
    }
    out["divide_ratio"] = (out["d_p90_pop"] / out["d_p10_pop"]
                           if out["d_p10_pop"] and out["d_p10_pop"] > 0 else np.nan)
    return pd.Series(out)


# ── national, latest quarter ────────────────────────────────────────────────
latest = df[(df.year == LATEST_Y) & (df.quarter == LATEST_Q)]
national = latest.groupby("service")[IND_COLS].apply(indicators)
for s in SERVICES:
    if s in national.index:
        cov = cells.loc[cells[f"measured_{s}"], "pop"].sum()
        national.loc[s, "pop_covered"]     = cov
        national.loc[s, "pop_coverage_pct"] = 100 * cov / POP_TOTAL
        national.loc[s, "area_coverage_pct"] = 100 * national.loc[s, "area_km2"] / COUNTRY_AREA_KM2

display(style_table(
    national[["tiles", "tests", "pop_coverage_pct", "area_coverage_pct",
              "d_median_tile", "d_median_pop", "d_mean_pop",
              "pct_above_bb", "pct_above_hi", "divide_ratio", "lat_median_pop"]]
    .rename(columns={
        "tiles": "Tiles", "tests": "Tests", "pop_coverage_pct": "Pop. covered %",
        "area_coverage_pct": "Area covered %", "d_median_tile": "Median (per tile)",
        "d_median_pop": "Median (per person)", "d_mean_pop": "Mean (per person)",
        "pct_above_bb": f"% pop ≥ {BROADBAND_MBPS} Mbps",
        "pct_above_hi": f"% pop ≥ {GOOD_SPEED_MBPS} Mbps",
        "divide_ratio": "P90/P10", "lat_median_pop": "Latency (ms)"})
    .style.format({"Tiles": "{:,.0f}", "Tests": "{:,.0f}", "Pop. covered %": "{:.1f}",
                   "Area covered %": "{:.2f}", "Median (per tile)": "{:.1f}",
                   "Median (per person)": "{:.1f}", "Mean (per person)": "{:.1f}",
                   f"% pop ≥ {BROADBAND_MBPS} Mbps": "{:.1f}",
                   f"% pop ≥ {GOOD_SPEED_MBPS} Mbps": "{:.1f}",
                   "P90/P10": "{:.1f}", "Latency (ms)": "{:.0f}"}),
    f"National connectivity indicators — {COUNTRY_NAME}, {LATEST_Y} Q{LATEST_Q} "
    f"(download speed, Mbps)"))

In [ ]:
# ── subnational table ───────────────────────────────────────────────────────
sub = (latest.dropna(subset=["admin_name"])
             .groupby(["service", "admin_name"])[IND_COLS]
             .apply(indicators).reset_index())

pop_adm = cells.groupby("admin_name")["pop"].sum().rename("pop_total").reset_index()

cov_rows = []
for s in SERVICES:
    c = (cells[cells[f"measured_{s}"]].groupby("admin_name")["pop"].sum()
         .rename("pop_covered").reset_index())
    c["service"] = s
    cov_rows.append(c)
sub = sub.merge(pd.concat(cov_rows, ignore_index=True),
                on=["service", "admin_name"], how="left")
sub["pop_covered"] = sub["pop_covered"].fillna(0.0)
sub = sub.merge(pop_adm, on="admin_name", how="left")
sub = sub.merge(admin[["admin_name", "area_km2"]].rename(columns={"area_km2": "adm_km2"}),
                on="admin_name", how="left")
sub["pop_coverage_pct"]  = 100 * sub["pop_covered"] / sub["pop_total"]
sub["area_coverage_pct"] = 100 * sub["area_km2"] / sub["adm_km2"]
sub["density"]           = sub["pop_total"] / sub["adm_km2"]

MAIN_SERVICE = "mobile" if "mobile" in SERVICES else SERVICES[0]
main = sub[sub.service == MAIN_SERVICE].sort_values("d_median_pop", ascending=False)

show = main[["admin_name", "pop_total", "pop_coverage_pct", "tests",
             "d_median_pop", "pct_above_bb", "divide_ratio", "lat_median_pop"]].copy()
show.columns = [ADMIN_LEVEL, "Population", "Pop. covered %", "Tests",
                "Median Mbps (per person)", f"% pop ≥ {BROADBAND_MBPS} Mbps",
                "P90/P10", "Latency ms"]
display(style_table(show.head(20).style.format({
    "Population": "{:,.0f}", "Pop. covered %": "{:.1f}", "Tests": "{:,.0f}",
    "Median Mbps (per person)": "{:.1f}", f"% pop ≥ {BROADBAND_MBPS} Mbps": "{:.1f}",
    "P90/P10": "{:.1f}", "Latency ms": "{:.0f}"}),
    f"{MAIN_SERVICE.capitalize()} connectivity by {ADMIN_LEVEL} — "
    f"{LATEST_Y} Q{LATEST_Q} (top 20 by population-weighted median)"))

In [ ]:
# ── settlement type (urban / peri-urban / rural) ────────────────────────────
settle = (latest.groupby(["service", "settlement"])[IND_COLS]
                .apply(indicators).reset_index())
pop_settle = cells.groupby("settlement")["pop"].sum().rename("pop_total").reset_index()
settle = settle.merge(pop_settle, on="settlement", how="left")
for s in SERVICES:
    for st in settle["settlement"].unique():
        m = cells[(cells.settlement == st) & (cells[f"measured_{s}"])]["pop"].sum()
        settle.loc[(settle.service == s) & (settle.settlement == st), "pop_covered"] = m
settle["pop_coverage_pct"] = 100 * settle["pop_covered"] / settle["pop_total"]

order = ["Urban", "Peri-urban", "Rural"]
settle["settlement"] = pd.Categorical(settle["settlement"], order, ordered=True)
settle = settle.sort_values(["service", "settlement"])

display(style_table(
    settle[settle.service == MAIN_SERVICE][
        ["settlement", "pop_total", "pop_coverage_pct", "tiles",
         "d_median_pop", "pct_above_bb"]]
    .rename(columns={"settlement": "Settlement type", "pop_total": "Population",
                     "pop_coverage_pct": "Pop. covered %", "tiles": "Tiles measured",
                     "d_median_pop": "Median Mbps (per person)",
                     "pct_above_bb": f"% pop ≥ {BROADBAND_MBPS} Mbps"})
    .style.format({"Population": "{:,.0f}", "Pop. covered %": "{:.1f}",
                   "Tiles measured": "{:,.0f}", "Median Mbps (per person)": "{:.1f}",
                   f"% pop ≥ {BROADBAND_MBPS} Mbps": "{:.1f}"}),
    f"The urban–rural divide — {MAIN_SERVICE}, {LATEST_Y} Q{LATEST_Q} "
    f"(urban ≥ {URBAN_DENS_MIN}/km², rural < {PERIURBAN_DENS_MIN}/km²)"))

# 11 · Bilingual output

The notebook itself stays in English, but the **published dashboard is bilingual**: every label,
title, insight, caption, tooltip and methodological note exists in English and in French, and the
reader switches between them with one click. The choice is remembered in the browser, and the page
opens in the reader's own language when it can detect it.

Everything translatable is gathered in the single dictionary below. Adding a third language means
adding one key to `LANGS` and one block to `TXT` — nothing else in the notebook changes.

> **A word on why this matters here.** A connectivity indicator that exists only in English is an
> indicator that half of the continent's statistical offices cannot circulate internally. Producing
> both versions from the same computation also guarantees that the two language versions can never
> drift apart: there is one number, rendered twice.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Bilingual string catalogue — every piece of text that reaches the reader
# ══════════════════════════════════════════════════════════════════════════════
LANGS = ["en", "fr"]          # add a language here + a block in TXT below
NB_LANG = "en"                # language used for the previews inside the notebook

TXT = {
"en": {
  # ---- chrome -------------------------------------------------------------
  "lang_name": "English", "other_lang_btn": "Français",
  "kicker": "AfDB · AU STATAFRIC · STG17 &nbsp;·&nbsp; {period}",
  "title": "{country} — Connectivity and Population",
  "subtitle": ("Ookla® Speedtest Open Data joined to WorldPop gridded population — "
               "a population-weighted connectivity indicator for every {level} unit"),
  "meta_desc": ("Population-weighted connectivity indicators for {country} from Ookla Speedtest "
                "Open Data and WorldPop, {period}."),
  "nav_kpi": "Key figures", "nav_ins": "Insights", "nav_maps": "Maps",
  "nav_charts": "Charts", "nav_table": "Table", "nav_method": "Method &amp; limitations",
  # ---- section 1 ----------------------------------------------------------
  "s1_kick": "01 · Key figures", "s1_title": "{country} at a glance",
  "s1_lede": ("{tests} {service} tests recorded in {tiles} measured squares, weighted by the "
              "population of {pop} inhabitants."),
  "s1_src": ("Reference period {period}. Figures “per person” are weighted by WorldPop {wpyear} "
             "gridded population; figures “per tile” give every measured square equal weight."),
  "kpi_pop": "Population", "kpi_med_person": "Median download<br>per person",
  "kpi_med_tile": "Median download<br>per tile", "kpi_above": "Population ≥ {bb} Mbps",
  "kpi_pop_meas": "Population measured", "kpi_area_meas": "Land area measured",
  "kpi_gini": "Connectivity Gini", "kpi_lat": "Median latency",
  "u_mbps": "Mbps", "u_ms": "ms", "u_pct": "%", "u_pct_meas": "% of measured", "u_wp": "WorldPop {y}",
  # ---- section 2 ----------------------------------------------------------
  "s2_kick": "02 · What the data says", "s2_title": "Automated findings",
  "s2_lede": "Generated directly from the computation — no figure in this section was typed by hand.",
  # ---- section 3 ----------------------------------------------------------
  "s3_kick": "03 · Geography", "s3_title": "Where connectivity is, and where measurement is not",
  "s3_lede": ("The first map answers “how fast”, the second “where exactly was it measured”, and "
              "the third — the one that usually changes the conversation — “who is missing from "
              "the sample”."),
  "map1_cap": "Map 1 — Population-weighted median download speed by {level}.",
  "map2_cap": ("Map 2 — The measurement grid itself. Use the layer control, top right, "
               "for test density."),
  "map3_cap": ("Map 3 — Populated cells with no measurement in the reference quarter. "
               "A sampling map, not a network coverage map."),
  # ---- section 4 ----------------------------------------------------------
  "s4_kick": "04 · Analysis", "s4_title": "Six views of the same question",
  "s4_lede": "Hover any chart for the underlying values.",
  # ---- section 5 ----------------------------------------------------------
  "s5_kick": "05 · Reference", "s5_title": "Indicators by {level}",
  "s5_lede": ("Also available as CSV and GeoJSON in this repository. “Per person” is the figure to "
              "quote; “per tile” is shown so that the difference stays visible."),
  "th_unit": "{level}", "th_pop": "Population", "th_meas": "Pop. measured", "th_tests": "Tests",
  "th_med_person": "Median Mbps<br>per person", "th_med_tile": "Median Mbps<br>per tile",
  "th_above": "% pop ≥ {bb} Mbps", "th_ratio": "P90/P10", "th_lat": "Latency ms",
  # ---- section 6 ----------------------------------------------------------
  "s6_kick": "06 · Documentation", "s6_title": "Method, sources and limitations",
  "h_sources": "Sources", "h_method": "Method",
  "h_limits": "Limitations — read before quoting any figure", "h_repro": "Reproducibility",
  "src_ookla": ("<b>Ookla® Speedtest Open Data</b> — performance tiles, Web-Mercator zoom 16 "
                "(≈611 m at the equator), {period}. Licence <b>CC BY-NC-SA 4.0</b>."),
  "src_wp": ("<b>WorldPop</b> — global gridded population {wpyear}, {wpres}, UN-adjusted. "
             "Licence CC BY 4.0."),
  "src_gb": ("<b>geoBoundaries</b> (gbOpen) — administrative boundaries {level}. "
             "Licence CC BY 4.0."),
  "meth_1": ("Tiles extracted from the global quarterly parquet files by quadkey range predicate, "
             "then clipped to the national polygon on the tile centroid."),
  "meth_2": ("Each tile is located in the WorldPop grid by its centroid; the population of a cell "
             "is shared equally among the tiles it contains, giving each tile a population weight."),
  "meth_3": ("Headline speeds are <b>population-weighted medians</b>: the median of the "
             "distribution in which each tile carries the weight of the people it represents."),
  "meth_4": ("Settlement classes are a density proxy: urban ≥ {urb} people/km², "
             "peri-urban ≥ {peri}, rural below."),
  "meth_5": ("Coverage of measurement = share of the national population living in a 1 km cell "
             "containing at least one measured tile."),
  "repro_1": ("Produced by the notebook "
              "<code>02-Lab-Ookla-Speedtest-Open-Data-and-WorldPop.ipynb</code>, "
              "generated on {date} for <b>{country} ({iso3})</b>."),
  "repro_2": ("Re-run it with a different <code>COUNTRY_ISO3</code> to rebuild the whole product "
              "for another country."),
  "footer": ("<b>{country} — Connectivity and population</b><br>"
             "African Development Bank · AU STATAFRIC — STG17.<br>"
             "Contains information from <b>Ookla® Speedtest Open Data</b>, used under "
             "<a href='https://creativecommons.org/licenses/by-nc-sa/4.0/'>CC BY-NC-SA 4.0</a>. "
             "Ookla trademarks are the property of Ookla, LLC; this product is not endorsed by or "
             "affiliated with Ookla. Derivative works must carry the same licence and must not be "
             "used commercially.<br>"
             "Population data © WorldPop (CC BY 4.0) · Boundaries © geoBoundaries (CC BY 4.0)."),
  # ---- charts -------------------------------------------------------------
  "src_note": ("Source: Ookla® Speedtest Open Data ({period}, CC BY-NC-SA 4.0) · "
               "WorldPop {wpyear} ({wpres}, CC BY 4.0) · geoBoundaries gbOpen"),
  "c1_title": "How fast is a measured square?",
  "c1_sub": ("Distribution of tile-level download speeds, 99th percentile clipped. "
             "The long right tail is why we publish medians."),
  "c1_x": "Download speed (Mbps)", "c1_y": "Number of tiles", "c1_tiles": "tiles",
  "c2_title": "Where is {service} connectivity strongest?",
  "c2_sub": "Population-weighted median download speed by {level}, {period}",
  "c2_x": "Population-weighted median download speed (Mbps)", "c2_nat": "national",
  "c2_h": ("Median (per person)", "Population", "Population measured", "Tests"),
  "c3_title": "Does density buy speed?",
  "c3_sub": "Each bubble is one {level} unit; size = population, colour = share of population measured",
  "c3_x": "Population density (people / km², log scale)", "c3_y": "Median download speed (Mbps)",
  "c3_cbar": "Pop.<br>measured %", "c3_h": ("Density", "Median", "Population"),
  "c3_r": "(log density)",
  "c4_title": "Is connectivity improving?",
  "c4_sub": "Population-weighted median speed (lines) and measurement volume (bars)",
  "c4_y": "Population-weighted median (Mbps)", "c4_y2": "Tiles measured",
  "c4_line": "{service} — median Mbps", "c4_bar": "{service} — tiles measured",
  "c5_title": "How unequally is bandwidth distributed?",
  "c5_sub": ("Connectivity Lorenz curve — Gini coefficient <b>{gini}</b> "
             "(0 = every person experiences the same speed, 1 = total concentration)"),
  "c5_x": "Cumulative share of the measured population (%)",
  "c5_y": "Cumulative share of measured bandwidth (%)",
  "c5_eq": "perfect equality", "c5_obs": "observed",
  "c5_h": "Poorest-served %{x:.0f}% of people<br>hold %{y:.0f}% of measured bandwidth",
  "c5_a": "<b>{v}%</b> of bandwidth<br>for {q}% of people",
  "c6_title": "Two divides, not one",
  "c6_sub": ("Speed gap (bars) and measurement gap (dotted line) between urban, peri-urban "
             "and rural areas"),
  "c6_y": "Median download speed (Mbps)", "c6_y2": "% of population measured",
  "c6_cov": "% of population measured",
  # ---- maps ---------------------------------------------------------------
  "mp_layer_speed": "{service} median speed", "mp_layer_grid": "{service} speed grid",
  "mp_layer_heat": "Test density (heatmap)", "mp_boundary": "National boundary",
  "mp_cbar1": "Population-weighted median {service} download speed (Mbps)",
  "mp_cbar2": "Download speed (Mbps) — {cell}",
  "mp_native": "native z16 tiles (≈611 m)", "mp_agg": "aggregated to z{z} (≈{km} km)",
  "mp_t_unit": "{level}:", "mp_t_pop": "Population:", "mp_t_meas": "Population measured (%):",
  "mp_t_person": "Median Mbps (per person):", "mp_t_tile": "Median Mbps (per tile):",
  "mp_t_above": "% pop ≥ {bb} Mbps:", "mp_t_lat": "Latency (ms):", "mp_t_tests": "Tests:",
  "mp_t_speed": "Download (Mbps):", "mp_t_estpop": "Est. population:", "mp_t_z16": "z16 tiles:",
  "mp_m1_title": "{service} connectivity by {level}",
  "mp_m3_legend": "Measurement gap", "mp_m3_a": "Dense unmeasured population",
  "mp_m3_b": "Unmeasured population", "mp_m3_c": "Measured population (hidden layer)",
  "mp_m3_note": ("{pop} people live in a 1 km cell with no Speedtest measurement in {period}."),
  "mp_layer_gap": "Unmeasured population", "mp_layer_meas": "Measured population",
  # ---- settlement classes -------------------------------------------------
  "Urban": "Urban", "Peri-urban": "Peri-urban", "Rural": "Rural",
},

"fr": {
  # ---- chrome -------------------------------------------------------------
  "lang_name": "Français", "other_lang_btn": "English",
  "kicker": "BAD · UA STATAFRIC · STG17 &nbsp;·&nbsp; {period}",
  "title": "{country} — Connectivité et population",
  "subtitle": ("Données ouvertes Ookla® Speedtest croisées avec la grille de population WorldPop — "
               "un indicateur de connectivité pondéré par la population pour chaque unité {level}"),
  "meta_desc": ("Indicateurs de connectivité pondérés par la population pour {country}, à partir "
                "des données ouvertes Ookla Speedtest et de WorldPop, {period}."),
  "nav_kpi": "Chiffres clés", "nav_ins": "Enseignements", "nav_maps": "Cartes",
  "nav_charts": "Graphiques", "nav_table": "Tableau", "nav_method": "Méthode et limites",
  # ---- section 1 ----------------------------------------------------------
  "s1_kick": "01 · Chiffres clés", "s1_title": "{country} en un coup d’œil",
  "s1_lede": ("{tests} tests {service} enregistrés dans {tiles} carreaux mesurés, pondérés par la "
              "population de {pop} habitants."),
  "s1_src": ("Période de référence {period}. Les valeurs « par habitant » sont pondérées par la "
             "grille de population WorldPop {wpyear} ; les valeurs « par carreau » donnent le même "
             "poids à chaque carreau mesuré."),
  "kpi_pop": "Population", "kpi_med_person": "Débit descendant<br>médian par habitant",
  "kpi_med_tile": "Débit descendant<br>médian par carreau", "kpi_above": "Population ≥ {bb} Mbit/s",
  "kpi_pop_meas": "Population mesurée", "kpi_area_meas": "Territoire mesuré",
  "kpi_gini": "Gini de connectivité", "kpi_lat": "Latence médiane",
  "u_mbps": "Mbit/s", "u_ms": "ms", "u_pct": "%", "u_pct_meas": "% des personnes mesurées",
  "u_wp": "WorldPop {y}",
  # ---- section 2 ----------------------------------------------------------
  "s2_kick": "02 · Ce que disent les données", "s2_title": "Enseignements générés automatiquement",
  "s2_lede": ("Produits directement par le calcul — aucun chiffre de cette section n’a été saisi "
              "à la main."),
  # ---- section 3 ----------------------------------------------------------
  "s3_kick": "03 · Géographie",
  "s3_title": "Où se trouve la connectivité, et où la mesure fait défaut",
  "s3_lede": ("La première carte répond à « à quelle vitesse », la deuxième à « où exactement la "
              "mesure a-t-elle eu lieu », et la troisième — celle qui change généralement la "
              "discussion — à « qui est absent de l’échantillon »."),
  "map1_cap": "Carte 1 — Débit descendant médian pondéré par la population, par unité {level}.",
  "map2_cap": ("Carte 2 — La grille de mesure elle-même. Le sélecteur de couches, en haut à "
               "droite, donne accès à la densité de tests."),
  "map3_cap": ("Carte 3 — Cellules peuplées sans aucune mesure sur le trimestre de référence. "
               "C’est une carte d’échantillonnage, pas une carte de couverture réseau."),
  # ---- section 4 ----------------------------------------------------------
  "s4_kick": "04 · Analyse", "s4_title": "Six regards sur la même question",
  "s4_lede": "Survolez un graphique pour afficher les valeurs sous-jacentes.",
  # ---- section 5 ----------------------------------------------------------
  "s5_kick": "05 · Référence", "s5_title": "Indicateurs par unité {level}",
  "s5_lede": ("Également disponibles en CSV et GeoJSON dans ce dépôt. La valeur « par habitant » "
              "est celle qu’il faut citer ; la valeur « par carreau » est affichée pour que "
              "l’écart reste visible."),
  "th_unit": "{level}", "th_pop": "Population", "th_meas": "Pop. mesurée", "th_tests": "Tests",
  "th_med_person": "Mbit/s médians<br>par habitant", "th_med_tile": "Mbit/s médians<br>par carreau",
  "th_above": "% pop ≥ {bb} Mbit/s", "th_ratio": "P90/P10", "th_lat": "Latence ms",
  # ---- section 6 ----------------------------------------------------------
  "s6_kick": "06 · Documentation", "s6_title": "Méthode, sources et limites",
  "h_sources": "Sources", "h_method": "Méthode",
  "h_limits": "Limites — à lire avant de citer le moindre chiffre",
  "h_repro": "Reproductibilité",
  "src_ookla": ("<b>Ookla® Speedtest Open Data</b> — carreaux de performance, zoom 16 en "
                "projection Web-Mercator (≈611 m à l’équateur), {period}. "
                "Licence <b>CC BY-NC-SA 4.0</b>."),
  "src_wp": ("<b>WorldPop</b> — grille mondiale de population {wpyear}, {wpres}, ajustée aux "
             "estimations des Nations unies. Licence CC BY 4.0."),
  "src_gb": ("<b>geoBoundaries</b> (gbOpen) — limites administratives {level}. "
             "Licence CC BY 4.0."),
  "meth_1": ("Carreaux extraits des fichiers parquet trimestriels mondiaux par prédicat "
             "d’intervalle sur le quadkey, puis découpés sur le polygone national selon le "
             "centroïde du carreau."),
  "meth_2": ("Chaque carreau est localisé dans la grille WorldPop par son centroïde ; la "
             "population d’une cellule est répartie à parts égales entre les carreaux qu’elle "
             "contient, ce qui donne à chaque carreau un poids de population."),
  "meth_3": ("Les débits mis en avant sont des <b>médianes pondérées par la population</b> : la "
             "médiane de la distribution dans laquelle chaque carreau porte le poids des "
             "habitants qu’il représente."),
  "meth_4": ("Les classes d’habitat sont une approximation par la densité : urbain ≥ {urb} "
             "habitants/km², périurbain ≥ {peri}, rural en dessous."),
  "meth_5": ("Couverture de la mesure = part de la population nationale vivant dans une cellule "
             "de 1 km contenant au moins un carreau mesuré."),
  "repro_1": ("Produit par le notebook "
              "<code>02-Lab-Ookla-Speedtest-Open-Data-and-WorldPop.ipynb</code>, "
              "généré le {date} pour <b>{country} ({iso3})</b>."),
  "repro_2": ("Relancez-le avec un autre <code>COUNTRY_ISO3</code> pour reconstruire l’ensemble du "
              "produit pour un autre pays."),
  "footer": ("<b>{country} — Connectivité et population</b><br>"
             "Banque africaine de développement · UA STATAFRIC — STG17.<br>"
             "Contient des informations issues des <b>données ouvertes Ookla® Speedtest</b>, "
             "utilisées sous licence "
             "<a href='https://creativecommons.org/licenses/by-nc-sa/4.0/deed.fr'>CC BY-NC-SA 4.0</a>. "
             "Les marques Ookla sont la propriété d’Ookla, LLC ; ce produit n’est ni approuvé par "
             "Ookla ni affilié à Ookla. Toute œuvre dérivée doit porter la même licence et ne peut "
             "faire l’objet d’un usage commercial.<br>"
             "Données de population © WorldPop (CC BY 4.0) · Limites © geoBoundaries (CC BY 4.0)."),
  # ---- charts -------------------------------------------------------------
  "src_note": ("Sources : Ookla® Speedtest Open Data ({period}, CC BY-NC-SA 4.0) · "
               "WorldPop {wpyear} ({wpres}, CC BY 4.0) · geoBoundaries gbOpen"),
  "c1_title": "À quelle vitesse va un carreau mesuré ?",
  "c1_sub": ("Distribution des débits descendants par carreau, écrêtée au 99ᵉ centile. "
             "C’est cette longue queue à droite qui impose de publier des médianes."),
  "c1_x": "Débit descendant (Mbit/s)", "c1_y": "Nombre de carreaux", "c1_tiles": "carreaux",
  "c2_title": "Où la connectivité {service} est-elle la meilleure ?",
  "c2_sub": ("Débit descendant médian pondéré par la population, par unité {level}, {period}"),
  "c2_x": "Débit descendant médian pondéré par la population (Mbit/s)", "c2_nat": "national",
  "c2_h": ("Médiane (par habitant)", "Population", "Population mesurée", "Tests"),
  "c3_title": "La densité achète-t-elle du débit ?",
  "c3_sub": ("Chaque bulle est une unité {level} ; taille = population, "
             "couleur = part de la population mesurée"),
  "c3_x": "Densité de population (habitants / km², échelle logarithmique)",
  "c3_y": "Débit descendant médian (Mbit/s)",
  "c3_cbar": "Pop.<br>mesurée %", "c3_h": ("Densité", "Médiane", "Population"),
  "c3_r": "(densité en log)",
  "c4_title": "La connectivité progresse-t-elle ?",
  "c4_sub": ("Débit médian pondéré par la population (courbes) et volume de mesure (barres)"),
  "c4_y": "Médiane pondérée par la population (Mbit/s)", "c4_y2": "Carreaux mesurés",
  "c4_line": "{service} — Mbit/s médians", "c4_bar": "{service} — carreaux mesurés",
  "c5_title": "La bande passante est-elle répartie inégalement ?",
  "c5_sub": ("Courbe de Lorenz de la connectivité — coefficient de Gini <b>{gini}</b> "
             "(0 = tout le monde connaît le même débit, 1 = concentration totale)"),
  "c5_x": "Part cumulée de la population mesurée (%)",
  "c5_y": "Part cumulée de la bande passante mesurée (%)",
  "c5_eq": "égalité parfaite", "c5_obs": "observé",
  "c5_h": ("Les %{x:.0f}% les moins bien servis<br>détiennent %{y:.0f}% de la bande passante mesurée"),
  "c5_a": "<b>{v}%</b> de la bande passante<br>pour {q}% des habitants",
  "c6_title": "Deux fractures, et non une seule",
  "c6_sub": ("Écart de débit (barres) et écart de mesure (pointillés) entre zones urbaines, "
             "périurbaines et rurales"),
  "c6_y": "Débit descendant médian (Mbit/s)", "c6_y2": "% de la population mesurée",
  "c6_cov": "% de la population mesurée",
  # ---- maps ---------------------------------------------------------------
  "mp_layer_speed": "Débit médian {service}", "mp_layer_grid": "Grille des débits {service}",
  "mp_layer_heat": "Densité de tests (carte de chaleur)", "mp_boundary": "Frontière nationale",
  "mp_cbar1": "Débit descendant {service} médian pondéré par la population (Mbit/s)",
  "mp_cbar2": "Débit descendant (Mbit/s) — {cell}",
  "mp_native": "carreaux z16 natifs (≈611 m)", "mp_agg": "agrégés au z{z} (≈{km} km)",
  "mp_t_unit": "{level} :", "mp_t_pop": "Population :", "mp_t_meas": "Population mesurée (%) :",
  "mp_t_person": "Mbit/s médians (par habitant) :", "mp_t_tile": "Mbit/s médians (par carreau) :",
  "mp_t_above": "% pop ≥ {bb} Mbit/s :", "mp_t_lat": "Latence (ms) :", "mp_t_tests": "Tests :",
  "mp_t_speed": "Débit descendant (Mbit/s) :", "mp_t_estpop": "Population estimée :",
  "mp_t_z16": "Carreaux z16 :",
  "mp_m1_title": "Connectivité {service} par unité {level}",
  "mp_m3_legend": "Déficit de mesure", "mp_m3_a": "Population non mesurée, dense",
  "mp_m3_b": "Population non mesurée", "mp_m3_c": "Population mesurée (couche masquée)",
  "mp_m3_note": ("{pop} habitants vivent dans une cellule de 1 km sans aucune mesure Speedtest "
                 "sur {period}."),
  "mp_layer_gap": "Population non mesurée", "mp_layer_meas": "Population mesurée",
  # ---- settlement classes -------------------------------------------------
  "Urban": "Urbain", "Peri-urban": "Périurbain", "Rural": "Rural",
},
}


def T(key, lang, **kw):
    """Translate `key` into `lang` and fill its placeholders."""
    s = TXT[lang][key]
    return s.format(**kw) if kw else s


# values injected into almost every string
def ctx(lang, **extra):
    d = dict(country=COUNTRY_NAME, iso3=COUNTRY_ISO3, level=ADMIN_LEVEL,
             period=f"{LATEST_Y} Q{LATEST_Q}", service=MAIN_SERVICE.capitalize(),
             bb=BROADBAND_MBPS, wpyear=WORLDPOP_YEAR, wpres=WORLDPOP_RES,
             urb=URBAN_DENS_MIN, peri=PERIURBAN_DENS_MIN)
    d.update(extra)
    return d


missing = {l: [k for k in TXT["en"] if k not in TXT[l]] for l in LANGS}
for l, ks in missing.items():
    if ks:
        print(f"WARNING — {l} is missing {len(ks)} keys: {ks[:6]}")
print(f"Languages: {', '.join(TXT[l]['lang_name'] for l in LANGS)}  ·  "
      f"{len(TXT['en'])} strings each  ·  notebook previews in '{NB_LANG}'")

# 12 · Automated insights, in both languages

Everything below is **computed, not written**. Re-run the notebook for another country and the
sentences change accordingly. This is deliberate: an insight you can generate is an insight you can
audit, and it is the part your dashboard readers will actually read.

Each finding is produced in English and in French **from the same numbers**, so the two versions
can never contradict each other. Numbers are formatted according to the conventions of each
language — French uses the comma as decimal separator and a narrow space as thousands separator,
and puts a space before the percent sign. Getting that wrong is the detail that tells a reader the
French version was an afterthought.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Locale-aware number formatting
# ══════════════════════════════════════════════════════════════════════════════
NBSP = "\u202f"          # narrow no-break space, the French thousands separator

def num(x, d=1, lang="en"):
    """Format a number following the conventions of the target language."""
    if x is None or not np.isfinite(x):
        return "–"
    s = f"{x:,.{d}f}"
    if lang == "fr":
        s = s.replace(",", "\x00").replace(".", ",").replace("\x00", NBSP)
    return s

def pct(x, d=1, lang="en"):
    return num(x, d, lang) + ("{}%".format(NBSP) if lang == "fr" else "%")

def big(x, lang="en", d=1):
    """Compact form: 1.9 M / 1,9 M, 12.4 k / 12,4 k."""
    if x is None or not np.isfinite(x):
        return "–"
    a = abs(x)
    if a >= 1e9:
        return num(x / 1e9, d, lang) + (f"{NBSP}Md" if lang == "fr" else " bn")
    if a >= 1e6:
        return num(x / 1e6, d, lang) + (f"{NBSP}M" if lang == "fr" else " M")
    if a >= 1e3:
        return num(x / 1e3, d, lang) + (f"{NBSP}k" if lang == "fr" else " k")
    return num(x, d, lang)

def mbps(x, lang="en", d=1):
    return num(x, d, lang) + (f"{NBSP}Mbit/s" if lang == "fr" else " Mbps")

print("en:", big(1938472, "en"), "·", pct(13.54, 1, "en"), "·", mbps(24.7, "en"))
print("fr:", big(1938472, "fr"), "·", pct(13.54, 1, "fr"), "·", mbps(24.7, "fr"))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  The findings — one computation, two languages
# ══════════════════════════════════════════════════════════════════════════════
INSIGHTS = []

def add(kind, en_title, en_text, fr_title, fr_text):
    INSIGHTS.append({"kind": kind,
                     "en": {"title": en_title, "text": en_text},
                     "fr": {"title": fr_title, "text": fr_text}})

nat = national.loc[MAIN_SERVICE]
SVC_FR = {"mobile": "mobile", "fixed": "fixe"}[MAIN_SERVICE]

# ── 1 · the weighting gap ──────────────────────────────────────────────────
gap = nat["d_median_pop"] - nat["d_median_tile"]
rel = abs(gap) / max(nat["d_median_tile"], 1e-9) * 100
add("info",
    "Weighting changes the headline",
    f"The median {MAIN_SERVICE} download speed is <b>{mbps(nat['d_median_tile'])} per measured "
    f"tile</b> but <b>{mbps(nat['d_median_pop'])} per person</b> — a gap of "
    f"{'+' if gap > 0 else '−'}{mbps(abs(gap))} ({num(rel, 0)}%). "
    + ("Population weighting raises the figure: the better-served squares are also the more "
       "densely populated ones." if gap > 0 else
       "Population weighting lowers the figure: the fastest squares are sparsely populated, so a "
       "tile average flatters the national picture."),
    "La pondération change le chiffre à retenir",
    f"Le débit descendant médian {SVC_FR} est de <b>{mbps(nat['d_median_tile'], 'fr')} par carreau "
    f"mesuré</b> mais de <b>{mbps(nat['d_median_pop'], 'fr')} par habitant</b> — un écart de "
    f"{'+' if gap > 0 else '−'}{mbps(abs(gap), 'fr')} ({pct(rel, 0, 'fr')}). "
    + ("La pondération par la population relève le chiffre : les carreaux les mieux servis sont "
       "aussi les plus densément peuplés." if gap > 0 else
       "La pondération par la population abaisse le chiffre : les carreaux les plus rapides sont "
       "peu peuplés, si bien qu’une moyenne par carreau flatte la situation nationale."))

# ── 2 · what the source can and cannot say ─────────────────────────────────
unmeasured = POP_TOTAL * (1 - nat["pop_coverage_pct"] / 100)
add("warn",
    "What the source can and cannot tell you",
    f"Measurements exist for <b>{pct(nat['pop_coverage_pct'])} of the population</b> but only "
    f"<b>{pct(nat['area_coverage_pct'], 2)} of the land area</b>. Every figure in this dashboard "
    f"describes the measured population; the remaining {big(unmeasured)} inhabitants are outside "
    "the sample — they are not necessarily unconnected.",
    "Ce que la source peut dire, et ce qu’elle ne peut pas dire",
    f"Des mesures existent pour <b>{pct(nat['pop_coverage_pct'], 1, 'fr')} de la population</b> "
    f"mais seulement <b>{pct(nat['area_coverage_pct'], 2, 'fr')} du territoire</b>. Tous les "
    "chiffres de ce tableau de bord décrivent la population mesurée ; les "
    f"{big(unmeasured, 'fr')} habitants restants sont hors échantillon — ils ne sont pas pour "
    "autant dépourvus de connexion.")

# ── 3 · service quality ────────────────────────────────────────────────────
add("info",
    f"Service quality against the {BROADBAND_MBPS} Mbps reference",
    f"<b>{pct(nat['pct_above_bb'])}</b> of the measured population lives where {MAIN_SERVICE} "
    f"download speeds reach at least {BROADBAND_MBPS} Mbps, and <b>{pct(nat['pct_above_hi'])}</b> "
    f"reach {GOOD_SPEED_MBPS} Mbps. Median latency is {num(nat['lat_median_pop'], 0)} ms.",
    f"Qualité de service au regard du seuil de {BROADBAND_MBPS} Mbit/s",
    f"<b>{pct(nat['pct_above_bb'], 1, 'fr')}</b> de la population mesurée vit là où le débit "
    f"descendant {SVC_FR} atteint au moins {BROADBAND_MBPS} Mbit/s, et "
    f"<b>{pct(nat['pct_above_hi'], 1, 'fr')}</b> atteint {GOOD_SPEED_MBPS} Mbit/s. "
    f"La latence médiane est de {num(nat['lat_median_pop'], 0, 'fr')} ms.")

# ── 4 · subnational spread ─────────────────────────────────────────────────
if len(main) >= 2:
    top, bot = main.iloc[0], main.iloc[-1]
    ratio = top["d_median_pop"] / max(bot["d_median_pop"], 1e-9)
    add("warn" if ratio > 3 else "info",
        "Subnational spread",
        f"<b>{top['admin_name']}</b> records {mbps(top['d_median_pop'])} against "
        f"<b>{bot['admin_name']}</b> at {mbps(bot['d_median_pop'])} — a ratio of "
        f"<b>{num(ratio)}×</b> across {len(main)} {ADMIN_LEVEL} units. The within-country "
        f"P90/P10 ratio is {num(nat['divide_ratio'])}×.",
        "Écarts infranationaux",
        f"<b>{top['admin_name']}</b> enregistre {mbps(top['d_median_pop'], 'fr')} contre "
        f"<b>{bot['admin_name']}</b> à {mbps(bot['d_median_pop'], 'fr')} — un rapport de "
        f"<b>{num(ratio, 1, 'fr')}×</b> sur {len(main)} unités {ADMIN_LEVEL}. Le rapport "
        f"P90/P10 à l’intérieur du pays est de {num(nat['divide_ratio'], 1, 'fr')}×.")

# ── 5 · the urban-rural divide ─────────────────────────────────────────────
try:
    sm = settle[settle.service == MAIN_SERVICE].set_index("settlement")
    if "Urban" in sm.index and "Rural" in sm.index:
        u, r = sm.loc["Urban"], sm.loc["Rural"]
        sgap = u["d_median_pop"] / max(r["d_median_pop"], 1e-9)
        mgap = u["pop_coverage_pct"] / max(r["pop_coverage_pct"], 1e-9)
        add("risk" if sgap > 2 else "warn",
            "The urban–rural divide",
            f"Urban areas: <b>{mbps(u['d_median_pop'])}</b> with {pct(u['pop_coverage_pct'], 0)} "
            f"of their population measured. Rural areas: <b>{mbps(r['d_median_pop'])}</b> with "
            f"only {pct(r['pop_coverage_pct'], 0)} measured. The speed gap is "
            f"<b>{num(sgap)}×</b>; the <i>measurement</i> gap is {num(mgap)}× and is itself a "
            "finding.",
            "La fracture urbain–rural",
            f"Zones urbaines : <b>{mbps(u['d_median_pop'], 'fr')}</b>, avec "
            f"{pct(u['pop_coverage_pct'], 0, 'fr')} de leur population mesurée. Zones rurales : "
            f"<b>{mbps(r['d_median_pop'], 'fr')}</b>, avec seulement "
            f"{pct(r['pop_coverage_pct'], 0, 'fr')} de mesure. L’écart de débit est de "
            f"<b>{num(sgap, 1, 'fr')}×</b> ; l’écart de <i>mesure</i> est de "
            f"{num(mgap, 1, 'fr')}× et constitue en soi un résultat.")
except Exception:
    pass

# ── 6 · fixed versus mobile ────────────────────────────────────────────────
if {"fixed", "mobile"} <= set(national.index):
    f_, m_ = national.loc["fixed"], national.loc["mobile"]
    mobile_first = m_["pop_coverage_pct"] > 2 * f_["pop_coverage_pct"]
    add("info",
        "Fixed versus mobile",
        f"Fixed broadband reaches <b>{mbps(f_['d_median_pop'])}</b> per person against "
        f"<b>{mbps(m_['d_median_pop'])}</b> for mobile, but is measured for only "
        f"{pct(f_['pop_coverage_pct'])} of the population against {pct(m_['pop_coverage_pct'])} "
        "for mobile. In this country, mobile is "
        + ("the de facto access technology." if mobile_first
           else "widely complemented by fixed access."),
        "Fixe et mobile",
        f"Le haut débit fixe atteint <b>{mbps(f_['d_median_pop'], 'fr')}</b> par habitant contre "
        f"<b>{mbps(m_['d_median_pop'], 'fr')}</b> pour le mobile, mais n’est mesuré que pour "
        f"{pct(f_['pop_coverage_pct'], 1, 'fr')} de la population contre "
        f"{pct(m_['pop_coverage_pct'], 1, 'fr')} pour le mobile. Dans ce pays, le mobile est "
        + ("de fait la technologie d’accès." if mobile_first
           else "largement complété par l’accès fixe."))

# ── 7 · sampling concentration ─────────────────────────────────────────────
_d = latest[latest.service == MAIN_SERVICE]
conc = _d.nlargest(max(1, len(_d) // 100), "tests")["tests"].sum() / max(_d["tests"].sum(), 1) * 100
add("warn",
    "Sampling concentration",
    f"The busiest 1% of tiles carry <b>{pct(conc, 0)}</b> of all {MAIN_SERVICE} tests. Any "
    "national average is therefore dominated by a very small number of locations — a further "
    "argument for the population-weighted median.",
    "Concentration de l’échantillon",
    f"Le 1 % de carreaux les plus actifs concentrent <b>{pct(conc, 0, 'fr')}</b> de l’ensemble des "
    f"tests {SVC_FR}. Toute moyenne nationale est donc dominée par un très petit nombre de lieux — "
    "un argument de plus en faveur de la médiane pondérée par la population.")

# ── 8 · trend ──────────────────────────────────────────────────────────────
if len(QUARTERS) > 1:
    tr = (df[df.service == MAIN_SERVICE]
          .groupby(["year", "quarter"])[IND_COLS].apply(indicators).reset_index()
          .sort_values(["year", "quarter"]))
    first, last = tr.iloc[0], tr.iloc[-1]
    chg = (last["d_median_pop"] - first["d_median_pop"]) / max(first["d_median_pop"], 1e-9) * 100
    p0 = f"{int(first['year'])} Q{int(first['quarter'])}"
    p1 = f"{int(last['year'])} Q{int(last['quarter'])}"
    add("info",
        "Trend over the observed quarters",
        f"Between {p0} and {p1}, the population-weighted median {MAIN_SERVICE} speed moved from "
        f"{mbps(first['d_median_pop'])} to {mbps(last['d_median_pop'])} "
        f"(<b>{'+' if chg >= 0 else '−'}{num(abs(chg), 0)}%</b>), while the number of measured "
        f"tiles went from {num(first['tiles'], 0)} to {num(last['tiles'], 0)}. Read the two "
        "together: more tests in new places can lower the average without any network degrading.",
        "Évolution sur les trimestres observés",
        f"Entre {p0} et {p1}, le débit médian {SVC_FR} pondéré par la population est passé de "
        f"{mbps(first['d_median_pop'], 'fr')} à {mbps(last['d_median_pop'], 'fr')} "
        f"(<b>{'+' if chg >= 0 else '−'}{pct(abs(chg), 0, 'fr')}</b>), tandis que le nombre de "
        f"carreaux mesurés passait de {num(first['tiles'], 0, 'fr')} à "
        f"{num(last['tiles'], 0, 'fr')}. Il faut lire les deux ensemble : davantage de tests dans "
        "de nouveaux lieux peut faire baisser la moyenne sans qu’aucun réseau ne se dégrade.")
else:
    tr = None

for i in INSIGHTS:
    callout(f"<b>{i[NB_LANG]['title']}.</b> {i[NB_LANG]['text']}", i["kind"])
print(f"{len(INSIGHTS)} findings generated in {len(LANGS)} languages.")

# 13 · Charts

Six figures, each answering one question, and each **built once per language**. The computation
happens a single time; only the labels change, which is what guarantees that the English and French
versions of a chart can never show different numbers.

They are kept in `FIGS[lang][name]` so the dashboard can embed them without recomputing anything.

A note on chart discipline, since these will end up in an official publication: no chart junk, no
gridline clutter, no legend when a direct label will do, sources on the figure, and forecasts
flagged as forecasts. The palette is fixed by the institutional style guide and the colour carries
meaning — green for performance, ochre for caution, brick for risk — so it is never re-mapped from
one chart to the next.

In [ ]:
FIGS = {l: {} for l in LANGS}

def finish(fig, title, subtitle, lang, height=430):
    fig.update_layout(
        height=height,
        title=dict(text=f"<b>{title}</b>" +
                        (f"<br><span style='font-size:12px;color:{SLATE}'>{subtitle}</span>"
                         if subtitle else ""),
                   x=0.01, xanchor="left", y=0.94, font=dict(size=17)),
        margin=dict(l=60, r=30, t=(78 if subtitle else 60), b=70),
    )
    fig.add_annotation(text=f"<i>{T('src_note', lang, **ctx(lang))}</i>",
                       xref="paper", yref="paper", x=0, y=-0.20, showarrow=False,
                       align="left", font=dict(size=9, color=SLATE))
    return fig


def svc_label(service, lang):
    return {"en": {"fixed": "Fixed", "mobile": "Mobile"},
            "fr": {"fixed": "Fixe",  "mobile": "Mobile"}}[lang][service]


# ══════════════════════════════════════════════════════════════════════════════
#  FIG 1 · distribution of measured speeds
# ══════════════════════════════════════════════════════════════════════════════
def fig_distribution(lang):
    C = ctx(lang)
    fig = go.Figure()
    for i, s in enumerate(SERVICES):
        d = latest[latest.service == s]
        if d.empty:
            continue
        fig.add_trace(go.Histogram(
            x=d["d_mbps"].clip(upper=d["d_mbps"].quantile(0.99)),
            name=svc_label(s, lang), opacity=0.72, nbinsx=60,
            marker_color=[GREEN, TEAL][i % 2],
            hovertemplate="%{x:.0f} " + T("u_mbps", lang) + "<br>%{y} " +
                          T("c1_tiles", lang) + "<extra>" + svc_label(s, lang) + "</extra>"))
    fig.add_vline(x=BROADBAND_MBPS, line=dict(color=OCHRE, dash="dash", width=2),
                  annotation_text=f" {BROADBAND_MBPS} {T('u_mbps', lang)}",
                  annotation_position="top", annotation_font=dict(color=OCHRE, size=11))
    fig.update_layout(barmode="overlay", xaxis_title=T("c1_x", lang),
                      yaxis_title=T("c1_y", lang))
    return finish(fig, T("c1_title", lang, **C), T("c1_sub", lang, **C), lang)


# ══════════════════════════════════════════════════════════════════════════════
#  FIG 2 · subnational ranking
# ══════════════════════════════════════════════════════════════════════════════
_rank = main.dropna(subset=["d_median_pop"]).sort_values("d_median_pop").tail(22)
_lo, _hi = float(_rank["d_median_pop"].min()), float(_rank["d_median_pop"].max())
_pos = (np.full(len(_rank), len(RAMP) // 2) if _hi <= _lo else
        np.interp(_rank["d_median_pop"], (_lo, _hi), (0, len(RAMP) - 1)))
_RANKCOL = [RAMP[min(len(RAMP) - 1, int(v))] for v in _pos]

def fig_ranking(lang):
    C = ctx(lang)
    h = T("c2_h", lang)
    fig = go.Figure(go.Bar(
        x=_rank["d_median_pop"], y=_rank["admin_name"], orientation="h",
        marker_color=_RANKCOL, marker_line=dict(width=0),
        text=[num(v, 1, lang) for v in _rank["d_median_pop"]], textposition="outside",
        textfont=dict(size=11, color=INK),
        customdata=np.stack([_rank["pop_total"], _rank["pop_coverage_pct"], _rank["tests"]], -1),
        hovertemplate=(f"<b>%{{y}}</b><br>{h[0]}: %{{x:.1f}} {T('u_mbps', lang)}"
                       f"<br>{h[1]}: %{{customdata[0]:,.0f}}"
                       f"<br>{h[2]}: %{{customdata[1]:.1f}}%"
                       f"<br>{h[3]}: %{{customdata[2]:,.0f}}<extra></extra>")))
    fig.add_vline(x=float(national.loc[MAIN_SERVICE, "d_median_pop"]),
                  line=dict(color=INK, dash="dot", width=1.5),
                  annotation_text=" " + T("c2_nat", lang), annotation_position="top",
                  annotation_font=dict(size=10, color=INK))
    fig.update_layout(xaxis_title=T("c2_x", lang), yaxis_title="",
                      yaxis=dict(tickfont=dict(size=11)))
    return finish(fig, T("c2_title", lang, **C), T("c2_sub", lang, **C), lang,
                  height=max(430, 22 * len(_rank) + 170))


# ══════════════════════════════════════════════════════════════════════════════
#  FIG 3 · density versus speed
# ══════════════════════════════════════════════════════════════════════════════
_sc = main.dropna(subset=["d_median_pop", "density"])
if len(_sc) > 3:
    _lx = np.log10(_sc["density"].clip(lower=1))
    _b, _a = np.polyfit(_lx, _sc["d_median_pop"], 1)
    _r = float(np.corrcoef(_lx, _sc["d_median_pop"])[0, 1])
else:
    _lx = _b = _a = _r = None

def fig_density(lang):
    C = ctx(lang)
    h = T("c3_h", lang)
    fig = go.Figure(go.Scatter(
        x=_sc["density"], y=_sc["d_median_pop"], mode="markers+text",
        text=[n if p > _sc["pop_total"].quantile(0.80) else ""
              for n, p in zip(_sc["admin_name"], _sc["pop_total"])],
        textposition="top center", textfont=dict(size=9, color=SLATE),
        marker=dict(size=np.sqrt(_sc["pop_total"]) / np.sqrt(_sc["pop_total"]).max() * 42 + 7,
                    color=_sc["pop_coverage_pct"], colorscale=[[0, "#C9D6D0"], [1, DEEP]],
                    showscale=True, line=dict(width=1, color="white"),
                    colorbar=dict(title=dict(text=T("c3_cbar", lang), font=dict(size=10)),
                                  thickness=12, len=0.65)),
        customdata=np.stack([_sc["pop_total"], _sc["pop_coverage_pct"]], -1),
        hovertemplate=(f"<b>%{{text}}</b><br>{h[0]}: %{{x:,.0f}} /km²<br>"
                       f"{h[1]}: %{{y:.1f}} {T('u_mbps', lang)}<br>"
                       f"{h[2]}: %{{customdata[0]:,.0f}}<extra></extra>")))
    if _lx is not None:
        xs = np.linspace(_lx.min(), _lx.max(), 50)
        fig.add_trace(go.Scatter(x=10 ** xs, y=_a + _b * xs, mode="lines",
                                 line=dict(color=OCHRE, dash="dash", width=2),
                                 hoverinfo="skip", showlegend=False))
        fig.add_annotation(x=0.98, y=0.06, xref="paper", yref="paper", showarrow=False,
                           text=f"<b>r = {num(_r, 2, lang)}</b> {T('c3_r', lang)}",
                           font=dict(size=12, color=OCHRE))
    fig.update_layout(xaxis=dict(title=T("c3_x", lang), type="log"),
                      yaxis_title=T("c3_y", lang))
    return finish(fig, T("c3_title", lang, **C), T("c3_sub", lang, **C), lang)


for l in LANGS:
    FIGS[l]["distribution"] = fig_distribution(l)
    FIGS[l]["ranking"] = fig_ranking(l)
    FIGS[l]["density"] = fig_density(l)

FIGS[NB_LANG]["distribution"].show()
FIGS[NB_LANG]["ranking"].show()
FIGS[NB_LANG]["density"].show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  FIG 4 · quarterly trend
# ══════════════════════════════════════════════════════════════════════════════
if len(QUARTERS) > 1:
    trend = (df.groupby(["service", "year", "quarter"])[IND_COLS]
               .apply(indicators).reset_index().sort_values(["year", "quarter"]))
    trend["period"] = trend["year"].astype(str) + " Q" + trend["quarter"].astype(str)

    def fig_trend(lang):
        C = ctx(lang)
        fig = go.Figure()
        for i, s in enumerate(SERVICES):
            t = trend[trend.service == s]
            if t.empty:
                continue
            sl = svc_label(s, lang)
            fig.add_trace(go.Scatter(
                x=t["period"], y=t["d_median_pop"], name=T("c4_line", lang, service=sl),
                mode="lines+markers+text", line=dict(color=[GREEN, TEAL][i % 2], width=3),
                marker=dict(size=9), text=[num(v, 0, lang) for v in t["d_median_pop"]],
                textposition="top center", textfont=dict(size=10, color=SLATE)))
            fig.add_trace(go.Bar(x=t["period"], y=t["tiles"],
                                 name=T("c4_bar", lang, service=sl),
                                 marker_color=[MINT, "#DCEDEF"][i % 2], yaxis="y2", opacity=0.9))
        fig.update_layout(
            barmode="group", yaxis=dict(title=T("c4_y", lang)),
            yaxis2=dict(title=T("c4_y2", lang), overlaying="y", side="right", showgrid=False,
                        tickfont=dict(color=SLATE, size=10)),
            legend=dict(orientation="h", y=1.10, x=0))
        return finish(fig, T("c4_title", lang, **C), T("c4_sub", lang, **C), lang)

    for l in LANGS:
        FIGS[l]["trend"] = fig_trend(l)
    FIGS[NB_LANG]["trend"].show()
else:
    trend = None
    print("Set N_QUARTERS > 1 in the configuration cell to produce the trend chart.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  FIG 5 · connectivity Lorenz curve
# ══════════════════════════════════════════════════════════════════════════════
_dl = latest[latest.service == MAIN_SERVICE].dropna(subset=["d_mbps", "pop_tile"])
_dl = _dl[_dl["pop_tile"] > 0].sort_values("d_mbps")
CUM_POP = np.cumsum(_dl["pop_tile"].values) / _dl["pop_tile"].sum() * 100
_cap = np.cumsum(_dl["pop_tile"].values * _dl["d_mbps"].values)
CUM_CAP = _cap / _cap[-1] * 100
_integ = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
gini = float(1 - 2 * _integ(CUM_CAP / 100, CUM_POP / 100))

def fig_lorenz(lang):
    C = ctx(lang, gini=num(gini, 3, lang))
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=[0, 100], y=[0, 100], mode="lines", name=T("c5_eq", lang),
                             line=dict(color=SAGE, dash="dash", width=2), hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=CUM_POP, y=CUM_CAP, mode="lines", name=T("c5_obs", lang),
                             line=dict(color=DEEP, width=3), fill="tonexty",
                             fillcolor="rgba(0,168,106,0.13)",
                             hovertemplate=T("c5_h", lang) + "<extra></extra>"))
    for q in (50, 80):
        y = float(np.interp(q, CUM_POP, CUM_CAP))
        fig.add_annotation(x=q, y=y, ax=35, ay=-35, arrowhead=0, arrowcolor=OCHRE,
                           text=T("c5_a", lang, v=num(y, 0, lang), q=q),
                           font=dict(size=10, color=OCHRE), bgcolor="white",
                           bordercolor=OCHRE, borderwidth=1, borderpad=4)
    fig.update_layout(xaxis_title=T("c5_x", lang), yaxis_title=T("c5_y", lang),
                      legend=dict(orientation="h", y=1.08, x=0))
    return finish(fig, T("c5_title", lang, **C), T("c5_sub", lang, **C), lang)


# ══════════════════════════════════════════════════════════════════════════════
#  FIG 6 · settlement type × service
# ══════════════════════════════════════════════════════════════════════════════
_sw = settle.dropna(subset=["d_median_pop"])

def fig_settlement(lang):
    C = ctx(lang)
    fig = go.Figure()
    for i, s in enumerate(SERVICES):
        t = _sw[_sw.service == s]
        if t.empty:
            continue
        sl = svc_label(s, lang)
        fig.add_trace(go.Bar(
            x=[T(str(v), lang) for v in t["settlement"]], y=t["d_median_pop"], name=sl,
            marker_color=[GREEN, TEAL][i % 2],
            text=[num(v, 1, lang) for v in t["d_median_pop"]], textposition="outside",
            customdata=np.stack([t["pop_total"], t["pop_coverage_pct"]], -1),
            hovertemplate=(f"<b>%{{x}}</b> — {sl}<br>%{{y:.1f}} {T('u_mbps', lang)}<br>"
                           f"{T('th_pop', lang)}: %{{customdata[0]:,.0f}}<br>"
                           f"{T('th_meas', lang)}: %{{customdata[1]:.1f}}%<extra></extra>")))
    t = _sw[_sw.service == MAIN_SERVICE]
    fig.add_trace(go.Scatter(
        x=[T(str(v), lang) for v in t["settlement"]], y=t["pop_coverage_pct"],
        name=T("c6_cov", lang), mode="lines+markers", yaxis="y2",
        line=dict(color=OCHRE, width=2.5, dash="dot"), marker=dict(size=10, symbol="diamond")))
    fig.update_layout(barmode="group", yaxis_title=T("c6_y", lang),
                      yaxis2=dict(title=T("c6_y2", lang), overlaying="y", side="right",
                                  range=[0, 100], showgrid=False,
                                  tickfont=dict(color=OCHRE, size=10)),
                      legend=dict(orientation="h", y=1.10, x=0))
    return finish(fig, T("c6_title", lang, **C), T("c6_sub", lang, **C), lang)


for l in LANGS:
    FIGS[l]["lorenz"] = fig_lorenz(l)
    FIGS[l]["settlement"] = fig_settlement(l)

FIGS[NB_LANG]["lorenz"].show()
FIGS[NB_LANG]["settlement"].show()
print(f"Connectivity Gini ({MAIN_SERVICE}, {LATEST_Y} Q{LATEST_Q}): {gini:.3f}")

# 14 · Interactive maps

Three layers, each built once per language so that tooltips, legends and layer names follow the
reader:

1. **Choropleth** — population-weighted median speed by administrative unit, with the full
   indicator set in the tooltip.
2. **Measurement grid** — the Ookla tiles themselves, aggregated to a coarser quadkey level when
   there are too many to draw; browsers stop being pleasant above roughly 15 000 polygons.
3. **Gap map** — where people live *without* any measurement. This is usually the most
   policy-relevant layer, and the one nobody produces.

In [ ]:
CENTER = [(BBOX[1] + BBOX[3]) / 2, (BBOX[0] + BBOX[2]) / 2]
SPAN = max(BBOX[2] - BBOX[0], BBOX[3] - BBOX[1])
ZOOM0 = int(max(4, min(10, round(8.5 - math.log2(max(SPAN, 0.05)) * 0.9))))
TIP_STYLE = (f"background:white;border:1px solid {SAGE};border-radius:4px;padding:8px;"
             f"font-family:{FONT};font-size:12px;color:{INK};"
             "box-shadow:0 2px 6px rgba(0,0,0,.15)")

def base_map(zoom=None, tiles="CartoDB positron"):
    m = folium.Map(location=CENTER, zoom_start=zoom or ZOOM0, tiles=tiles,
                   control_scale=True, prefer_canvas=True)
    Fullscreen(position="topright").add_to(m)
    return m

def legend_html(title, entries, note=""):
    rows = "".join(
        f'<div style="display:flex;align-items:center;gap:7px;margin:2px 0">'
        f'<span style="width:15px;height:11px;background:{c};display:inline-block;'
        f'border:1px solid rgba(0,0,0,.18)"></span><span>{l}</span></div>'
        for c, l in entries)
    return folium.Element(f"""
    <div style="position:fixed;bottom:22px;left:14px;z-index:9999;background:white;
                border:1px solid {SAGE};border-radius:5px;padding:9px 12px;
                font-family:{FONT};font-size:11.5px;color:{INK};
                box-shadow:0 2px 6px rgba(0,0,0,.14);max-width:235px">
      <div style="font-weight:700;font-size:10px;letter-spacing:1.6px;
                  text-transform:uppercase;color:{DEEP};margin-bottom:5px">{title}</div>
      {rows}
      <div style="color:{SLATE};font-size:10px;margin-top:5px;line-height:1.4">{note}</div>
    </div>""")

MAPS = {l: {} for l in LANGS}
print(f"Map centre {CENTER[0]:.2f}, {CENTER[1]:.2f} · initial zoom {ZOOM0}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  MAP 1 · administrative choropleth
# ══════════════════════════════════════════════════════════════════════════════
amap = admin.merge(
    main[["admin_name", "d_median_pop", "d_median_tile", "pop_total", "pop_coverage_pct",
          "pct_above_bb", "tests", "tiles", "divide_ratio", "lat_median_pop", "density"]],
    on="admin_name", how="left")

_vals = amap["d_median_pop"].dropna()
_qs = (np.unique(np.quantile(_vals, np.linspace(0, 1, len(RAMP) + 1)))
       if len(_vals) >= 2 else np.array([]))

def admin_colormap(lang):
    if len(_qs) >= 3:
        c = cm.StepColormap(RAMP[:len(_qs) - 1], index=list(_qs),
                            vmin=float(_qs[0]), vmax=float(_qs[-1]))
    else:
        c = cm.LinearColormap(RAMP, vmin=0,
                              vmax=max(1, float(_vals.max() if len(_vals) else 1)))
    c.caption = T("mp_cbar1", lang, service=svc_label(MAIN_SERVICE, lang))
    return c

def build_map_admin(lang):
    C = ctx(lang, service=svc_label(MAIN_SERVICE, lang))
    cmap = admin_colormap(lang)

    def style(feat):
        v = feat["properties"].get("d_median_pop")
        ok = v is not None and np.isfinite(v)
        return {"fillColor": cmap(v) if ok else "#E4E9E6",
                "color": "white", "weight": 1.1, "fillOpacity": 0.86}

    m = base_map()
    folium.GeoJson(
        amap.to_json(), name=T("mp_layer_speed", lang, service=svc_label(MAIN_SERVICE, lang)),
        style_function=style,
        highlight_function=lambda f: {"weight": 3, "color": GOLD, "fillOpacity": 0.95},
        tooltip=folium.GeoJsonTooltip(
            fields=["admin_name", "pop_total", "pop_coverage_pct", "d_median_pop",
                    "d_median_tile", "pct_above_bb", "lat_median_pop", "tests"],
            aliases=[T("mp_t_unit", lang, level=ADMIN_LEVEL), T("mp_t_pop", lang),
                     T("mp_t_meas", lang), T("mp_t_person", lang), T("mp_t_tile", lang),
                     T("mp_t_above", lang, bb=BROADBAND_MBPS), T("mp_t_lat", lang),
                     T("mp_t_tests", lang)],
            localize=True, sticky=False, style=TIP_STYLE)).add_to(m)
    cmap.add_to(m)
    folium.GeoJson(adm0.to_json(), name=T("mp_boundary", lang),
                   style_function=lambda f: {"color": INK, "weight": 2, "fill": False}).add_to(m)
    folium.LayerControl(collapsed=True).add_to(m)
    m.get_root().html.add_child(folium.Element(
        f'<div style="position:fixed;top:12px;left:60px;z-index:9999;background:white;'
        f'border-left:4px solid {GREEN};padding:8px 14px;font-family:{FONT};'
        f'box-shadow:0 2px 6px rgba(0,0,0,.14);border-radius:0 4px 4px 0">'
        f'<div style="font-size:9.5px;letter-spacing:2px;font-weight:700;color:{DEEP}">'
        f'{COUNTRY_NAME.upper()} · {LATEST_Y} Q{LATEST_Q}</div>'
        f'<div style="font-size:14px;font-weight:700;color:{INK}">'
        f'{T("mp_m1_title", lang, **C)}</div></div>'))
    return m

for l in LANGS:
    MAPS[l]["admin"] = build_map_admin(l)
MAPS[NB_LANG]["admin"]

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  MAP 2 · the measurement grid itself
# ══════════════════════════════════════════════════════════════════════════════
tl = latest[latest.service == MAIN_SERVICE].copy()
agg_zoom = 16
while len(tl["quadkey"].str[:agg_zoom].unique()) > MAX_MAP_TILES and agg_zoom > 10:
    agg_zoom -= 1

if agg_zoom < 16:
    tl["qk"] = tl["quadkey"].str[:agg_zoom]
    tl["_w"] = tl["tests"].clip(lower=1)
    tl["_wd"] = tl["d_mbps"] * tl["_w"]
    g = (tl.groupby("qk").agg(_wd=("_wd", "sum"), _w=("_w", "sum"), tests=("tests", "sum"),
                              pop=("pop_tile", "sum"), n=("quadkey", "size")).reset_index())
    g["d_mbps"] = g["_wd"] / g["_w"]
    grid = pd.concat([g, quadkeys_to_bounds(g["qk"].values, zoom=agg_zoom)], axis=1)
    CELL_LABEL = {l: T("mp_agg", l, z=agg_zoom,
                       km=f"{611 * 2 ** (16 - agg_zoom) / 1000:.1f}") for l in LANGS}
else:
    grid = tl.rename(columns={"pop_tile": "pop"}).copy()
    grid["n"] = 1
    CELL_LABEL = {l: T("mp_native", l) for l in LANGS}

_gv = grid["d_mbps"].dropna()
_br = np.unique(np.quantile(_gv, np.linspace(0, 1, len(RAMP) + 1)))

GRID_FEATURES = [{
    "type": "Feature",
    "geometry": {"type": "Polygon", "coordinates": [[
        [r.west, r.south], [r.east, r.south], [r.east, r.north],
        [r.west, r.north], [r.west, r.south]]]},
    "properties": {"speed": round(float(r.d_mbps), 1), "tests": int(r.tests),
                   "pop": int(r.pop), "n": int(r.n)}} for r in grid.itertuples()]

def build_map_grid(lang):
    sl = svc_label(MAIN_SERVICE, lang)
    if len(_br) >= 3:
        tmap = cm.StepColormap(RAMP[:len(_br) - 1], index=list(_br),
                               vmin=float(_br[0]), vmax=float(_br[-1]))
    else:
        tmap = cm.LinearColormap(RAMP, vmin=float(_gv.min()), vmax=float(max(_gv.max(), 1)))
    tmap.caption = T("mp_cbar2", lang, cell=CELL_LABEL[lang])
    m = base_map(zoom=ZOOM0)
    folium.GeoJson(
        {"type": "FeatureCollection", "features": GRID_FEATURES},
        name=T("mp_layer_grid", lang, service=sl),
        style_function=lambda f: {"fillColor": tmap(f["properties"]["speed"]),
                                  "color": "none", "fillOpacity": 0.78},
        tooltip=folium.GeoJsonTooltip(
            fields=["speed", "tests", "pop", "n"],
            aliases=[T("mp_t_speed", lang), T("mp_t_tests", lang),
                     T("mp_t_estpop", lang), T("mp_t_z16", lang)],
            style=TIP_STYLE)).add_to(m)
    HeatMap([[r.lat, r.lon, float(r.tests)] for r in grid.itertuples()],
            name=T("mp_layer_heat", lang), radius=13, blur=18, min_opacity=0.25,
            gradient={0.2: "#E8F5EF", 0.5: GREEN, 0.8: DEEP, 1.0: FOREST},
            show=False).add_to(m)
    folium.GeoJson(adm0.to_json(), name=T("mp_boundary", lang),
                   style_function=lambda f: {"color": INK, "weight": 1.6, "fill": False}).add_to(m)
    tmap.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m

for l in LANGS:
    MAPS[l]["grid"] = build_map_grid(l)
print(f"{len(GRID_FEATURES):,} polygons drawn · {CELL_LABEL[NB_LANG]}")
MAPS[NB_LANG]["grid"]

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  MAP 3 · the measurement gap — where people live without any test
# ══════════════════════════════════════════════════════════════════════════════
gap_cells = cells[(~cells["measured_any"]) & (cells["pop"] > 0)]
meas_cells = cells[cells["measured_any"] & (cells["pop"] > 0)]
GAP_POP = float(gap_cells["pop"].sum())
_gap_top = gap_cells.nlargest(min(6000, len(gap_cells)), "pop")
_meas_top = meas_cells.nlargest(min(6000, len(meas_cells)), "pop")

def build_map_gap(lang):
    m = base_map(tiles="CartoDB dark_matter")
    folium.GeoJson(adm0.to_json(), name=T("mp_boundary", lang),
                   style_function=lambda f: {"color": "#FFFFFF", "weight": 1.4,
                                             "fill": False, "opacity": 0.6}).add_to(m)
    HeatMap([[r.lat, r.lon, float(r.pop)] for r in _gap_top.itertuples()],
            name=T("mp_layer_gap", lang), radius=12, blur=16, min_opacity=0.32,
            gradient={0.2: "#F6D58A", 0.5: OCHRE, 0.8: TERRA, 1.0: BRICK}).add_to(m)
    HeatMap([[r.lat, r.lon, float(r.pop)] for r in _meas_top.itertuples()],
            name=T("mp_layer_meas", lang), radius=11, blur=15, min_opacity=0.28,
            gradient={0.2: "#9ED9C0", 0.6: GREEN, 1.0: "#00FFB0"}, show=False).add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    m.get_root().html.add_child(legend_html(
        T("mp_m3_legend", lang),
        [(BRICK, T("mp_m3_a", lang)), (OCHRE, T("mp_m3_b", lang)), (GREEN, T("mp_m3_c", lang))],
        T("mp_m3_note", lang, pop=big(GAP_POP, lang), period=f"{LATEST_Y} Q{LATEST_Q}")))
    return m

for l in LANGS:
    MAPS[l]["gap"] = build_map_gap(l)

kpi_row([("Population with no measurement", big(GAP_POP), "", BRICK),
         ("Share of national population", f"{100 * GAP_POP / POP_TOTAL:.1f}", "%", OCHRE),
         ("Unmeasured populated cells", f"{len(gap_cells):,}", "", SLATE)])
callout("Read this map as a <b>sampling</b> map, not a coverage map. It shows where Ookla has no "
        "observation — which may mean no network, no smartphone, no data bundle, or simply no "
        "reason to run a test. Distinguishing between those four requires household survey or "
        "operator data, and that distinction is the honest limit of this product.", "risk")
MAPS[NB_LANG]["gap"]

# 15 · The dashboard

Everything computed above is assembled into **one self-contained HTML file** carrying both language
versions, with a switch in the top-right corner. No build step, no server, no framework: open it
locally, email it, or push it to a repository and it becomes a public URL. That property matters
more than it sounds — it is what makes the output survivable after the training, when nobody is
left to maintain a Node application.

**How the switch works.** Both language versions are rendered into the same document, inside
`<div class="lang-block" data-lang="…">`. A single CSS rule shows one and hides the other, and four
lines of JavaScript flip the attribute, remember the choice in the browser, and tell Plotly to
resize the charts that have just become visible. The page opens in French automatically for a
reader whose browser is set to French. There is no round-trip to a server and no dependency on a
translation service — the file works offline, from a USB stick if need be.

**What goes in:** the KPI strip, the automated findings, the three interactive maps, the six
charts, the subnational table, and a methodology block with the licences and the limitations
statement — all of it twice.

**What does not:** anything that needs a key, a login or a paid service.

In [ ]:
from string import Template
import html as _html

def fig_html(fig, div_id):
    return pio.to_html(fig, full_html=False, include_plotlyjs=False, div_id=div_id,
                       config={"displayModeBar": False, "responsive": True})

def map_html(m, height=560):
    """Embed a folium map as a fully self-contained responsive iframe."""
    raw = m.get_root().render()
    return (f'<iframe srcdoc="{_html.escape(raw, quote=True)}" loading="lazy" '
            f'style="width:100%;height:{height}px;border:1px solid {SAGE};'
            f'border-radius:6px;background:white"></iframe>')


# ══════════════════════════════════════════════════════════════════════════════
#  Per-language content blocks
# ══════════════════════════════════════════════════════════════════════════════
nat = national.loc[MAIN_SERVICE]

def kpi_block(lang):
    items = [
        (T("kpi_pop", lang),        big(POP_TOTAL, lang),
         T("u_wp", lang, y=WORLDPOP_YEAR), GREEN),
        (T("kpi_med_person", lang), num(nat["d_median_pop"], 1, lang), T("u_mbps", lang), DEEP),
        (T("kpi_med_tile", lang),   num(nat["d_median_tile"], 1, lang), T("u_mbps", lang), SLATE),
        (T("kpi_above", lang, bb=BROADBAND_MBPS), num(nat["pct_above_bb"], 0, lang),
         T("u_pct_meas", lang), TEAL),
        (T("kpi_pop_meas", lang),   num(nat["pop_coverage_pct"], 0, lang), T("u_pct", lang), OCHRE),
        (T("kpi_area_meas", lang),  num(nat["area_coverage_pct"], 2, lang), T("u_pct", lang), TERRA),
        (T("kpi_gini", lang),       num(gini, 2, lang), "0–1", BRICK),
        (T("kpi_lat", lang),        num(nat["lat_median_pop"], 0, lang), T("u_ms", lang), INK),
    ]
    return "".join(f"""
      <div class="kpi"><div class="kpi-l">{lab}</div>
        <div class="kpi-v" style="color:{col}">{val}<span class="kpi-u">{unit}</span></div>
      </div>""" for lab, val, unit, col in items)


ICOL = {"info": GREEN, "warn": OCHRE, "risk": BRICK}
IBG = {"info": MINT, "warn": "#FDF4E0", "risk": "#FBECEA"}

def insights_block(lang):
    return "".join(f"""
      <div class="ins" style="background:{IBG[i['kind']]};border-color:{ICOL[i['kind']]}">
        <div class="ins-t" style="color:{ICOL[i['kind']]}">{i[lang]['title']}</div>
        <div class="ins-b">{i[lang]['text']}</div>
      </div>""" for i in INSIGHTS)


def table_block(lang):
    rows = []
    for r in main.itertuples():
        if not np.isfinite(r.d_median_pop):
            continue
        rows.append(
            "<tr>"
            f"<td class='nm'>{r.admin_name}</td>"
            f"<td>{num(r.pop_total, 0, lang)}</td>"
            f"<td>{pct(r.pop_coverage_pct, 1, lang)}</td>"
            f"<td>{num(r.tests, 0, lang)}</td>"
            f"<td class='hi'>{num(r.d_median_pop, 1, lang)}</td>"
            f"<td>{num(r.d_median_tile, 1, lang)}</td>"
            f"<td>{pct(r.pct_above_bb, 0, lang)}</td>"
            f"<td>{num(r.divide_ratio, 1, lang)}×</td>"
            f"<td>{num(r.lat_median_pop, 0, lang)}</td></tr>")
    return "".join(rows)


def limitations(lang):
    unmeasured = 100 - nat["pop_coverage_pct"]
    if lang == "en":
        return (
            "<b>1 · Speedtest data is crowdsourced and self-selected.</b> A tile exists because "
            "somebody chose to run a test there — typically because they suspected a problem or had "
            "just changed connection. It is not a probability sample and carries no design weights."
            f"<br><br><b>2 · Absence of measurement is not absence of service.</b> "
            f"{pct(unmeasured, 0)} of the population lives in a cell with no test this quarter; "
            "the map of gaps is a map of Speedtest users, not of network coverage."
            "<br><br><b>3 · Device and tariff effects are not separable.</b> A slow measurement may "
            "reflect an old handset, an exhausted data bundle or a congested cell, not the network."
            f"<br><br><b>4 · Only {pct(nat['area_coverage_pct'], 2)} of the land area is "
            "measured</b>, and the busiest 1% of tiles carry a large share of all tests, so "
            "unweighted national averages are dominated by a few urban locations."
            "<br><br><b>5 · Population figures are modelled.</b> WorldPop redistributes census or "
            "projected counts using covariates including built-up area and night-time lights; it is "
            "not a census and should not be used to validate another light-derived indicator."
            "<br><br><b>6 · Boundaries are from geoBoundaries</b>, not from the national mapping "
            "authority; small differences in geometry will move the subnational figures."
            "<br><br><b>7 · Licence.</b> The Ookla licence is non-commercial and share-alike. This "
            "product and any derivative must carry CC BY-NC-SA 4.0 and may not be sold or embedded "
            "in a commercial service. Confirm with your legal service before an official release.")
    return (
        "<b>1 · Les données Speedtest sont produites par les utilisateurs et auto-sélectionnées.</b> "
        "Un carreau existe parce que quelqu’un a choisi d’y lancer un test — le plus souvent parce "
        "qu’il soupçonnait un problème ou venait de changer de connexion. Ce n’est pas un "
        "échantillon probabiliste et il ne comporte aucune pondération de sondage."
        f"<br><br><b>2 · L’absence de mesure n’est pas l’absence de service.</b> "
        f"{pct(unmeasured, 0, 'fr')} de la population vit dans une cellule sans aucun test ce "
        "trimestre ; la carte des lacunes est une carte des utilisateurs de Speedtest, pas de la "
        "couverture réseau."
        "<br><br><b>3 · Les effets du terminal et du forfait ne sont pas séparables.</b> Une mesure "
        "faible peut refléter un téléphone ancien, un forfait épuisé ou une cellule congestionnée, "
        "et non le réseau disponible."
        f"<br><br><b>4 · Seuls {pct(nat['area_coverage_pct'], 2, 'fr')} du territoire sont "
        "mesurés</b>, et le 1 % de carreaux les plus actifs concentrent une large part des tests : "
        "les moyennes nationales non pondérées sont dominées par quelques lieux urbains."
        "<br><br><b>5 · Les chiffres de population sont modélisés.</b> WorldPop redistribue des "
        "effectifs censitaires ou projetés à l’aide de covariables incluant le bâti et les lumières "
        "nocturnes ; ce n’est pas un recensement et il ne doit pas servir à valider un autre "
        "indicateur dérivé des lumières nocturnes."
        "<br><br><b>6 · Les limites administratives proviennent de geoBoundaries</b>, et non de "
        "l’autorité cartographique nationale ; de légères différences de géométrie déplacent les "
        "valeurs infranationales."
        "<br><br><b>7 · Licence.</b> La licence Ookla est non commerciale et à partage dans les "
        "mêmes conditions. Ce produit et toute œuvre dérivée doivent porter la licence "
        "CC BY-NC-SA 4.0 et ne peuvent être vendus ni intégrés à un service commercial. Faites "
        "confirmer ce point par votre service juridique avant toute diffusion officielle.")

print("Content builders ready.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Stylesheet (institutional palette, AfDB-inspired)
# ══════════════════════════════════════════════════════════════════════════════
CSS = Template(r"""
  :root{--green:$GREEN;--deep:$DEEP;--forest:$FOREST;--gold:$GOLD;--ochre:$OCHRE;
        --ink:$INK;--slate:$SLATE;--mist:$MIST;--mint:$MINT;--sage:$SAGE;--brick:$BRICK;}
  *{box-sizing:border-box}
  body{margin:0;background:#fff;color:var(--ink);
       font-family:Calibri,'Segoe UI',Helvetica,Arial,sans-serif;line-height:1.6}
  .wrap{max-width:1220px;margin:0 auto;padding:0 26px}
  .lang-block{display:none}
  html[data-lang="en"] .lang-block[data-lang="en"]{display:block}
  html[data-lang="fr"] .lang-block[data-lang="fr"]{display:block}
  #langbar{position:fixed;top:14px;right:18px;z-index:200;display:flex;gap:0;
           background:rgba(255,255,255,.14);border:1px solid rgba(255,255,255,.45);
           border-radius:20px;overflow:hidden;backdrop-filter:blur(4px)}
  #langbar button{border:0;background:transparent;color:#fff;font-weight:700;font-size:12px;
        letter-spacing:1.4px;padding:7px 15px;cursor:pointer;font-family:inherit}
  #langbar button.on{background:var(--gold);color:var(--ink)}
  header{background:linear-gradient(120deg,var(--forest) 0%,var(--deep) 45%,var(--green) 100%);
         color:#fff;padding:52px 0 44px;position:relative;overflow:hidden}
  header:after{content:"";position:absolute;right:-90px;top:-90px;width:330px;height:330px;
       border-radius:50%;border:42px solid rgba(255,255,255,.07)}
  .kick{font-size:11px;font-weight:700;letter-spacing:3.2px;text-transform:uppercase;
        color:var(--gold)}
  h1{font-size:40px;margin:11px 0 6px;font-weight:700;line-height:1.1}
  .sub{font-size:17px;font-style:italic;color:#E6F6EE;max-width:780px}
  .goldbar{height:5px;background:var(--gold);width:118px;margin-top:22px}
  nav{position:sticky;top:0;z-index:50;background:#fff;border-bottom:1px solid var(--sage);
      box-shadow:0 1px 4px rgba(0,0,0,.05)}
  nav .wrap{display:flex;gap:26px;overflow-x:auto;padding-top:13px;padding-bottom:13px}
  nav a{color:var(--slate);text-decoration:none;font-size:12.5px;font-weight:600;
        letter-spacing:.4px;white-space:nowrap;padding-bottom:2px;border-bottom:2px solid transparent}
  nav a:hover{color:var(--deep);border-bottom-color:var(--green)}
  section{padding:44px 0 8px}
  h2{font-size:26px;margin:0 0 6px}
  .h2k{font-size:10.5px;font-weight:700;letter-spacing:3px;text-transform:uppercase;
       color:var(--deep)}
  .lede{color:var(--slate);font-size:14px;max-width:880px;margin:0 0 22px}
  .kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(178px,1fr));gap:13px;
        margin:8px 0 6px}
  .kpi{background:#fff;border:1px solid var(--sage);border-radius:6px;padding:15px 17px}
  .kpi-l{font-size:9.5px;font-weight:700;letter-spacing:1.9px;text-transform:uppercase;
         color:var(--slate);line-height:1.35;min-height:26px}
  .kpi-v{font-size:33px;font-weight:700;line-height:1.14;margin-top:7px}
  .kpi-u{font-size:12.5px;font-weight:600;color:var(--slate);margin-left:5px}
  .ins-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(340px,1fr));gap:14px}
  .ins{border:1px solid;border-left-width:4px;border-radius:5px;padding:14px 17px}
  .ins-t{font-size:13.5px;font-weight:700;margin-bottom:5px}
  .ins-b{font-size:13px;color:var(--ink)}
  .card{background:#fff;border:1px solid var(--sage);border-radius:7px;padding:8px 10px;margin:16px 0}
  .grid2{display:grid;grid-template-columns:1fr 1fr;gap:18px}
  @media(max-width:900px){.grid2{grid-template-columns:1fr}h1{font-size:30px}}
  table{width:100%;border-collapse:collapse;font-size:12.5px}
  th{background:var(--deep);color:#fff;text-align:right;padding:9px 11px;font-weight:600;
     position:sticky;top:0;font-size:11.5px}
  th:first-child{text-align:left}
  td{padding:7px 11px;border-bottom:1px solid var(--sage);text-align:right}
  td.nm{text-align:left;font-weight:600}
  td.hi{color:var(--deep);font-weight:700}
  tbody tr:hover{background:var(--mint)}
  .tblwrap{max-height:520px;overflow:auto;border:1px solid var(--sage);border-radius:7px}
  .meth{background:var(--mist);border-radius:7px;padding:24px 28px;font-size:13.5px}
  .meth h3{margin:20px 0 7px;font-size:15px;color:var(--deep)}
  .meth h3:first-child{margin-top:0}
  .meth ul{margin:6px 0;padding-left:20px}
  .meth li{margin:4px 0}
  .warnbox{background:#FBECEA;border:1px solid var(--brick);border-left-width:4px;
           border-radius:5px;padding:15px 18px;font-size:13.5px;margin:16px 0}
  footer{background:var(--ink);color:#C9D2CD;font-size:12px;padding:32px 0;margin-top:52px}
  footer a{color:var(--gold);text-decoration:none}
  .src{font-size:10.5px;font-style:italic;color:var(--slate);margin-top:4px}
  code{background:var(--mint);padding:1px 5px;border-radius:3px;font-size:12px}
""").substitute(GREEN=GREEN, DEEP=DEEP, FOREST=FOREST, GOLD=GOLD, OCHRE=OCHRE,
                INK=INK, SLATE=SLATE, MIST=MIST, MINT=MINT, SAGE=SAGE, BRICK=BRICK)

LANG_JS = """
  function setLang(l){
    document.documentElement.setAttribute('data-lang', l);
    document.documentElement.setAttribute('lang', l);
    try{ localStorage.setItem('dashLang', l); }catch(e){}
    document.querySelectorAll('#langbar button').forEach(function(b){
      b.classList.toggle('on', b.dataset.lang === l); });
    setTimeout(function(){
      document.querySelectorAll('.lang-block[data-lang="'+l+'"] .js-plotly-plot')
        .forEach(function(p){ try{ Plotly.Plots.resize(p); }catch(e){} });
    }, 60);
  }
  (function(){
    var saved = null;
    try{ saved = localStorage.getItem('dashLang'); }catch(e){}
    var auto = (navigator.language || 'en').toLowerCase().indexOf('fr') === 0 ? 'fr' : null;
    setLang(saved || auto || DEFAULT_LANG);
  })();
"""
print(f"Stylesheet: {len(CSS):,} characters")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Assemble one full document containing every language
# ══════════════════════════════════════════════════════════════════════════════
def language_block(lang):
    C = ctx(lang)
    charts = {k: fig_html(FIGS[lang][k], f"fig-{k}-{lang}") for k in FIGS[lang]}
    trend_card = (f'<div class="card">{charts["trend"]}</div>' if "trend" in charts else "")
    th = [T("th_unit", lang, level=ADMIN_LEVEL), T("th_pop", lang), T("th_meas", lang),
          T("th_tests", lang), T("th_med_person", lang), T("th_med_tile", lang),
          T("th_above", lang, bb=BROADBAND_MBPS), T("th_ratio", lang), T("th_lat", lang)]
    return f"""
<div class="lang-block" data-lang="{lang}">

<header><div class="wrap">
  <div class="kick">{T("kicker", lang, **C)}</div>
  <h1>{T("title", lang, **C)}</h1>
  <div class="sub">{T("subtitle", lang, **C)}</div>
  <div class="goldbar"></div>
</div></header>

<nav><div class="wrap">
  <a href="#kpi-{lang}">{T("nav_kpi", lang)}</a>
  <a href="#ins-{lang}">{T("nav_ins", lang)}</a>
  <a href="#maps-{lang}">{T("nav_maps", lang)}</a>
  <a href="#charts-{lang}">{T("nav_charts", lang)}</a>
  <a href="#table-{lang}">{T("nav_table", lang)}</a>
  <a href="#method-{lang}">{T("nav_method", lang)}</a>
</div></nav>

<section id="kpi-{lang}"><div class="wrap">
  <div class="h2k">{T("s1_kick", lang)}</div><h2>{T("s1_title", lang, **C)}</h2>
  <p class="lede">{T("s1_lede", lang, tests=big(nat['tests'], lang),
                     tiles=num(nat['tiles'], 0, lang), pop=big(POP_TOTAL, lang),
                     service=svc_label(MAIN_SERVICE, lang).lower())}</p>
  <div class="kpis">{kpi_block(lang)}</div>
  <div class="src">{T("s1_src", lang, **C)}</div>
</div></section>

<section id="ins-{lang}"><div class="wrap">
  <div class="h2k">{T("s2_kick", lang)}</div><h2>{T("s2_title", lang)}</h2>
  <p class="lede">{T("s2_lede", lang)}</p>
  <div class="ins-grid">{insights_block(lang)}</div>
</div></section>

<section id="maps-{lang}"><div class="wrap">
  <div class="h2k">{T("s3_kick", lang)}</div><h2>{T("s3_title", lang)}</h2>
  <p class="lede">{T("s3_lede", lang)}</p>
  <div class="card">{map_html(MAPS[lang]["admin"])}</div>
  <div class="src">{T("map1_cap", lang, **C)}</div>
  <div class="card">{map_html(MAPS[lang]["grid"])}</div>
  <div class="src">{T("map2_cap", lang)}</div>
  <div class="card">{map_html(MAPS[lang]["gap"], 520)}</div>
  <div class="src">{T("map3_cap", lang)}</div>
</div></section>

<section id="charts-{lang}"><div class="wrap">
  <div class="h2k">{T("s4_kick", lang)}</div><h2>{T("s4_title", lang)}</h2>
  <p class="lede">{T("s4_lede", lang)}</p>
  <div class="card">{charts["distribution"]}</div>
  <div class="card">{charts["ranking"]}</div>
  <div class="grid2">
    <div class="card">{charts["density"]}</div>
    <div class="card">{charts["lorenz"]}</div>
  </div>
  <div class="card">{charts["settlement"]}</div>
  {trend_card}
</div></section>

<section id="table-{lang}"><div class="wrap">
  <div class="h2k">{T("s5_kick", lang)}</div><h2>{T("s5_title", lang, **C)}</h2>
  <p class="lede">{T("s5_lede", lang)}</p>
  <div class="tblwrap"><table>
    <thead><tr>{"".join(f"<th>{h}</th>" for h in th)}</tr></thead>
    <tbody>{table_block(lang)}</tbody>
  </table></div>
</div></section>

<section id="method-{lang}"><div class="wrap">
  <div class="h2k">{T("s6_kick", lang)}</div><h2>{T("s6_title", lang)}</h2>
  <div class="meth">
    <h3>{T("h_sources", lang)}</h3>
    <ul>
      <li>{T("src_ookla", lang, **C)}</li>
      <li>{T("src_wp", lang, **C)}</li>
      <li>{T("src_gb", lang, **C)}</li>
    </ul>
    <h3>{T("h_method", lang)}</h3>
    <ul>
      <li>{T("meth_1", lang)}</li><li>{T("meth_2", lang)}</li><li>{T("meth_3", lang)}</li>
      <li>{T("meth_4", lang, **C)}</li><li>{T("meth_5", lang)}</li>
    </ul>
    <h3>{T("h_limits", lang)}</h3>
    <div class="warnbox">{limitations(lang)}</div>
    <h3>{T("h_repro", lang)}</h3>
    <ul>
      <li>{T("repro_1", lang, date=dt.date.today().isoformat(), **C)}</li>
      <li>{T("repro_2", lang)}</li>
    </ul>
  </div>
</div></section>

<footer><div class="wrap">{T("footer", lang, **C)}</div></footer>
</div>"""


buttons = "".join(
    f'<button data-lang="{l}" onclick="setLang(\'{l}\')">{l.upper()}</button>' for l in LANGS)

html_out = f"""<!DOCTYPE html>
<html lang="{LANGS[0]}" data-lang="{LANGS[0]}"><head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>{T("title", LANGS[0], **ctx(LANGS[0]))}</title>
<meta name="description" content="{T("meta_desc", LANGS[0], **ctx(LANGS[0]))}">
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js" charset="utf-8"></script>
<style>{CSS}</style></head><body>
<div id="langbar">{buttons}</div>
{"".join(language_block(l) for l in LANGS)}
<script>const DEFAULT_LANG = "{LANGS[0]}";{LANG_JS}</script>
</body></html>"""

DASH_PATH = Path(OUTPUT_DIR) / "index.html"
DASH_PATH.write_text(html_out, encoding="utf-8")
size_mb = DASH_PATH.stat().st_size / 1e6

banner("DASHBOARD BUILT", f"{DASH_PATH}  ·  {size_mb:.1f} MB  ·  "
       f"{' / '.join(TXT[l]['lang_name'] for l in LANGS)}",
       "Single self-contained file carrying every language version. The reader switches with the "
       "buttons in the top-right corner; the choice is remembered, and the page opens in French by "
       "itself for a French-configured browser. The file name <code>index.html</code> is "
       "deliberate — a web server returns it at the root of a site.")
if size_mb > 45:
    callout(f"The dashboard weighs {size_mb:.0f} MB, which is heavy for a public page. Lower "
            "<code>MAX_MAP_TILES</code> in the configuration cell and rebuild, or drop one "
            "language from <code>LANGS</code>.", "warn")

In [ ]:
# Preview inside the notebook (scroll inside the frame, and try the EN/FR buttons)
display(HTML(f"""
<div style="border:1px solid {SAGE};border-radius:7px;overflow:hidden;margin-top:8px">
  <iframe src="{DASH_PATH.as_posix()}" style="width:100%;height:780px;border:0"></iframe>
</div>
<div style="font-family:{FONT};font-size:11.5px;color:{SLATE};margin-top:5px">
  If the frame stays blank — a Colab/Kaggle sandbox restriction — download
  <code>{DASH_PATH}</code> and open it locally. The file itself is complete.
</div>"""))

# 16 · Exports

The dashboard is the *communication* product. The files below are the *statistical* product — what
a peer NSO, a researcher or your own analysts will actually reuse. The last cell of this section
writes the bilingual `README.md` and the `LICENSE` that accompany them once the folder is published.

In [ ]:
stamp = f"{COUNTRY_ISO3}_{LATEST_Y}Q{LATEST_Q}"
out = Path(OUTPUT_DIR)
written = []

# 1 · subnational indicators (the headline deliverable) ----------------------
export_cols = ["service", "admin_name", "pop_total", "pop_covered", "pop_coverage_pct",
               "area_coverage_pct", "tiles", "tests", "devices", "d_median_pop", "d_mean_pop",
               "d_median_tile", "d_p10_pop", "d_p90_pop", "divide_ratio", "u_median_pop",
               "lat_median_pop", "pct_above_bb", "pct_above_hi", "density"]
sub_out = sub[[c for c in export_cols if c in sub.columns]].copy()
sub_out.insert(0, "iso3", COUNTRY_ISO3)
sub_out.insert(1, "period", f"{LATEST_Y}Q{LATEST_Q}")
sub_out.insert(2, "admin_level", ADMIN_LEVEL)
p = out / f"connectivity_{ADMIN_LEVEL}_{stamp}.csv"; sub_out.to_csv(p, index=False); written.append(p)

# 2 · same thing as GeoJSON, for QGIS / web maps ------------------------------
p = out / f"connectivity_{ADMIN_LEVEL}_{stamp}.geojson"
amap.to_file(p, driver="GeoJSON"); written.append(p)

# 3 · national summary --------------------------------------------------------
nat_out = national.reset_index()
nat_out.insert(0, "iso3", COUNTRY_ISO3); nat_out.insert(1, "period", f"{LATEST_Y}Q{LATEST_Q}")
p = out / f"national_summary_{stamp}.csv"; nat_out.to_csv(p, index=False); written.append(p)

# 4 · tile-level micro-file (parquet: 5-10x smaller than CSV) ----------------
tile_cols = ["quadkey", "service", "year", "quarter", "lon", "lat", "d_mbps", "u_mbps",
             "latency_ms", "tests", "devices", "pop_tile", "settlement", "admin_name"]
p = out / f"tiles_{stamp}.parquet"
df[[c for c in tile_cols if c in df.columns]].to_parquet(p, index=False); written.append(p)

# 5 · settlement + trend ------------------------------------------------------
p = out / f"settlement_{stamp}.csv"; settle.to_csv(p, index=False); written.append(p)
if trend is not None:
    p = out / f"trend_{COUNTRY_ISO3}.csv"; trend.to_csv(p, index=False); written.append(p)

# 6 · machine-readable metadata ----------------------------------------------
meta = {
    "title": {l: T("title", l, **ctx(l)) for l in LANGS},
    "iso3": COUNTRY_ISO3, "country": COUNTRY_NAME, "languages": LANGS,
    "reference_period": f"{LATEST_Y}Q{LATEST_Q}",
    "quarters_processed": [f"{y}Q{q}" for y, q in QUARTERS],
    "services": SERVICES, "admin_level": ADMIN_LEVEL,
    "generated": dt.datetime.now().isoformat(timespec="seconds"),
    "sources": {
        "ookla": {"name": "Ookla Speedtest Open Data", "licence": "CC BY-NC-SA 4.0",
                  "url": "https://github.com/teamookla/ookla-open-data",
                  "resolution": "Web Mercator z16 (~611 m at the equator)"},
        "worldpop": {"name": f"WorldPop {WORLDPOP_YEAR} {WORLDPOP_RES} UN-adjusted",
                     "licence": "CC BY 4.0", "url": "https://www.worldpop.org"},
        "geoboundaries": {"name": "geoBoundaries gbOpen", "licence": "CC BY 4.0",
                          "url": "https://www.geoboundaries.org"}},
    "methods": {"weighting": "population-weighted median; cell population shared equally "
                             "among the tiles whose centroid falls in the cell",
                "coverage": "share of population in 1 km cells containing >= 1 measured tile",
                "thresholds_mbps": {"broadband": BROADBAND_MBPS, "high_quality": GOOD_SPEED_MBPS},
                "settlement_density_thresholds": {"urban": URBAN_DENS_MIN,
                                                  "periurban": PERIURBAN_DENS_MIN}},
    "headline": {k: (float(nat[k]) if np.isfinite(nat[k]) else None)
                 for k in ["d_median_pop", "d_median_tile", "pct_above_bb", "pct_above_hi",
                           "pop_coverage_pct", "area_coverage_pct", "lat_median_pop"]},
    "population_total": POP_TOTAL, "connectivity_gini": float(gini),
    "licence_of_this_product": "CC BY-NC-SA 4.0 (inherited from Ookla, share-alike)",
}
p = out / "metadata.json"; p.write_text(json.dumps(meta, indent=2, ensure_ascii=False),
                                        encoding="utf-8"); written.append(p)

for f in written:
    print(f"  {f.stat().st_size/1024:9,.0f} KB   {f.name}")
print(f"\n{len(written) + 1} files in {out.resolve()} (dashboard included)")

In [ ]:
README = f"""# {COUNTRY_NAME} — Connectivity and Population · Connectivité et population

**Population-weighted connectivity indicators from Ookla® Speedtest Open Data and WorldPop**
**Indicateurs de connectivité pondérés par la population, à partir des données ouvertes Ookla® Speedtest et de WorldPop**

[![Dashboard](https://img.shields.io/badge/dashboard-live-00A86A)](./index.html)
![Licence](https://img.shields.io/badge/licence-CC%20BY--NC--SA%204.0-D49A00)
![Languages](https://img.shields.io/badge/langues-EN%20%7C%20FR-00704A)

African Development Bank · AU STATAFRIC — STG17.
Generated on {dt.date.today().isoformat()} for **{COUNTRY_NAME} ({COUNTRY_ISO3})**,
reference period **{LATEST_Y} Q{LATEST_Q}**. The dashboard is bilingual: use the EN / FR switch in
the top-right corner. *Le tableau de bord est bilingue : utilisez le sélecteur EN / FR en haut à droite.*

## Headline figures · Chiffres clés

| Indicator · Indicateur | Value · Valeur |
|---|---|
| Population ({WORLDPOP_YEAR}, WorldPop) | {POP_TOTAL:,.0f} |
| Median {MAIN_SERVICE} download **per person** · Débit médian **par habitant** | **{nat['d_median_pop']:.1f} Mbps** |
| Median download per measured tile · Débit médian par carreau mesuré | {nat['d_median_tile']:.1f} Mbps |
| Population ≥ {BROADBAND_MBPS} Mbps · Population ≥ {BROADBAND_MBPS} Mbit/s | {nat['pct_above_bb']:.1f}% |
| Population measured · Population mesurée | {nat['pop_coverage_pct']:.1f}% |
| Land area measured · Territoire mesuré | {nat['area_coverage_pct']:.2f}% |
| Connectivity Gini · Gini de connectivité | {gini:.3f} |
| Median latency · Latence médiane | {nat['lat_median_pop']:.0f} ms |
| {ADMIN_LEVEL} units · Unités {ADMIN_LEVEL} | {len(main)} |

## Contents · Contenu

| File | Description |
|---|---|
| `index.html` | Bilingual interactive dashboard · Tableau de bord interactif bilingue |
| `connectivity_{ADMIN_LEVEL}_{stamp}.csv` | Indicators by administrative unit · Indicateurs par unité administrative |
| `connectivity_{ADMIN_LEVEL}_{stamp}.geojson` | Same, with geometry · Idem, avec géométrie |
| `national_summary_{stamp}.csv` | National aggregates · Agrégats nationaux |
| `tiles_{stamp}.parquet` | Tile-level micro-file · Fichier détail au carreau |
| `settlement_{stamp}.csv` | Urban / peri-urban / rural · Urbain / périurbain / rural |
| `metadata.json` | Machine-readable provenance · Provenance lisible par machine |
| `.nojekyll` | Tells GitHub Pages to serve the files as they are |

## Method · Méthode

**EN.** Ookla publishes quarterly performance tiles at Web-Mercator zoom 16 (≈611 m at the equator).
Tiles covering {COUNTRY_NAME} were extracted directly from the global parquet files using a quadkey
range predicate, clipped to the national polygon on the tile centroid, and converted from kbps to
Mbps. Each tile was located in the WorldPop {WORLDPOP_YEAR} {WORLDPOP_RES} UN-adjusted population
grid by its centroid; the population of each grid cell was shared equally among the tiles it
contains, giving every tile a population weight. All headline speeds are **population-weighted
medians**. Coverage of measurement is the share of the national population living in a grid cell
that contains at least one measured tile. Settlement classes are a density proxy
(urban ≥ {URBAN_DENS_MIN} people/km², peri-urban ≥ {PERIURBAN_DENS_MIN}, rural below).

**FR.** Ookla publie des carreaux de performance trimestriels au zoom 16 en projection Web-Mercator
(≈611 m à l'équateur). Les carreaux couvrant {COUNTRY_NAME} ont été extraits directement des
fichiers parquet mondiaux au moyen d'un prédicat d'intervalle sur le quadkey, découpés sur le
polygone national selon le centroïde du carreau, puis convertis de kbit/s en Mbit/s. Chaque carreau
a été localisé dans la grille de population WorldPop {WORLDPOP_YEAR} {WORLDPOP_RES} ajustée aux
estimations des Nations unies ; la population de chaque cellule a été répartie à parts égales entre
les carreaux qu'elle contient, ce qui donne à chaque carreau un poids de population. Tous les débits
mis en avant sont des **médianes pondérées par la population**. La couverture de la mesure est la
part de la population nationale vivant dans une cellule contenant au moins un carreau mesuré. Les
classes d'habitat sont une approximation par la densité (urbain ≥ {URBAN_DENS_MIN} hab./km²,
périurbain ≥ {PERIURBAN_DENS_MIN}, rural en dessous).

## Limitations · Limites

**EN.** (1) Speedtest measurements are user-initiated and self-selected: no probability sample, no
design weights. (2) Absence of measurement is not absence of service —
{100 - nat['pop_coverage_pct']:.0f}% of the population lives in a cell with no test in the reference
quarter. (3) Device and tariff effects cannot be separated from network performance. (4) Only
{nat['area_coverage_pct']:.2f}% of the land area is measured and tests are heavily concentrated, so
unweighted national averages are biased upward. (5) WorldPop is a modelled surface, not a census,
and is partly built from night-time lights. (6) Boundaries come from geoBoundaries, not from the
national mapping authority. (7) This is **experimental statistics**, not an official indicator,
unless validated against operator or survey data.

**FR.** (1) Les mesures Speedtest sont lancées par les utilisateurs et auto-sélectionnées : ni
échantillon probabiliste, ni pondération de sondage. (2) L'absence de mesure n'est pas l'absence de
service — {100 - nat['pop_coverage_pct']:.0f} % de la population vit dans une cellule sans aucun
test sur le trimestre de référence. (3) Les effets du terminal et du forfait ne peuvent être séparés
de la performance du réseau. (4) Seuls {nat['area_coverage_pct']:.2f} % du territoire sont mesurés
et les tests sont très concentrés : les moyennes nationales non pondérées sont biaisées vers le
haut. (5) WorldPop est une surface modélisée, pas un recensement, et repose en partie sur les
lumières nocturnes. (6) Les limites administratives proviennent de geoBoundaries, et non de
l'autorité cartographique nationale. (7) Il s'agit de **statistiques expérimentales**, et non d'un
indicateur officiel, tant qu'elles n'ont pas été validées contre des données d'opérateurs ou
d'enquête.

## Sources and licences · Sources et licences

- **Ookla® Speedtest Open Data** — <https://github.com/teamookla/ookla-open-data> — **CC BY-NC-SA 4.0**
- **WorldPop** {WORLDPOP_YEAR} ({WORLDPOP_RES}, UN-adjusted) — <https://www.worldpop.org> — CC BY 4.0
- **geoBoundaries** (gbOpen) — <https://www.geoboundaries.org> — CC BY 4.0

## Licence of this product · Licence de ce produit

Released under **CC BY-NC-SA 4.0**, inherited from the Ookla licence (share-alike).
**Non-commercial use only.** Ookla trademarks are the property of Ookla, LLC. This product is not
endorsed by or affiliated with Ookla.

*Diffusé sous licence **CC BY-NC-SA 4.0**, héritée de la licence Ookla (partage dans les mêmes
conditions). **Usage non commercial uniquement.** Les marques Ookla sont la propriété d'Ookla, LLC.
Ce produit n'est ni approuvé par Ookla ni affilié à Ookla.*

## Reproducing · Reproduire

Open the notebook `02-Lab-Ookla-Speedtest-Open-Data-and-WorldPop.ipynb`, set
`COUNTRY_ISO3 = "{COUNTRY_ISO3}"` in the configuration cell, and run all cells. It runs unchanged in
Google Colab, Kaggle and locally, and requires no API key.
"""

LICENSE_TXT = """This work is licensed under the Creative Commons
Attribution-NonCommercial-ShareAlike 4.0 International License (CC BY-NC-SA 4.0).

Full text: https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode

The share-alike obligation is inherited from Ookla Speedtest Open Data, which is
distributed under CC BY-NC-SA 4.0 and from which this work is derived.

Contains information from Ookla Speedtest Open Data (c) Ookla, LLC.
Ookla trademarks are the property of Ookla, LLC. This product is not endorsed
by or affiliated with Ookla.

Population data (c) WorldPop, University of Southampton (CC BY 4.0).
Administrative boundaries (c) geoBoundaries, W. M. Geolab (CC BY 4.0).
"""

(out / "README.md").write_text(README, encoding="utf-8")
(out / "LICENSE").write_text(LICENSE_TXT, encoding="utf-8")
print(f"Written: README.md ({len(README):,} characters, bilingual) · LICENSE")

# 17 · Publishing — ask for a token, get a public URL

The dashboard is a single self-contained `index.html`, so any static host will serve it. We use
**GitHub Pages**: free, permanent, versioned, and no server to maintain.

**There is nothing to configure.** Run the two cells below. The second one asks for a GitHub token
in a hidden prompt, then does the rest by itself: it creates the **public** repository, uploads
every file, switches Pages on, waits for the first build and prints the live URL.

## Getting a token — 60 seconds, once

1. GitHub → your avatar → **Settings** → **Developer settings** → **Personal access tokens** →
   **Tokens (classic)** → *Generate new token (classic)*.
2. Tick the **`repo`** scope. Set an expiry; 30 days is plenty.
3. Copy the token. GitHub shows it **once**.

A fine-grained token also works: **Repository access → All repositories**, with **Read and write**
on *Contents*, *Pages* and *Administration*.

> **The token is never written into this notebook.** It is read from a hidden `getpass` prompt, or
> from the `GITHUB_TOKEN` environment variable, or — cleanest in Colab — from the **Secrets** panel
> (the key icon in the left sidebar; name the secret `GITHUB_TOKEN`). Nothing is printed, nothing
> is stored in a cell, nothing ends up in the committed `.ipynb`.

## What "public" means here

The repository is created **public**, and Pages then serves the dashboard to anyone, with no login
and no GitHub account required. That is the objective — and it is also the risk. The first cell
prints the exact list of files that are about to become world-readable: check it. It must contain
no microdata, no personal data and nothing under embargo.

In [ ]:
# A .nojekyll file tells GitHub Pages to serve the directory exactly as it is,
# instead of running it through Jekyll, which silently ignores some files.
(out / ".nojekyll").write_text("", encoding="utf-8")

PUBLISH_FILES = sorted(p for p in out.iterdir() if p.is_file())
total_mb = sum(p.stat().st_size for p in PUBLISH_FILES) / 1e6

print(f"{len(PUBLISH_FILES)} files will become publicly readable  ·  {total_mb:.1f} MB total\n")
for p in PUBLISH_FILES:
    flag = "   <-- this is the website" if p.name == "index.html" else ""
    print(f"  {p.stat().st_size/1024:9,.0f} KB   {p.name}{flag}")

if total_mb > 90:
    callout("Over 90 MB. GitHub rejects individual files above 100 MB and Pages sites above 1 GB. "
            "Lower <code>MAX_MAP_TILES</code> in the configuration cell and rebuild the dashboard, "
            "or leave the tile-level parquet out of the publication.", "warn")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Publication to GitHub Pages
# ══════════════════════════════════════════════════════════════════════════════
import base64, getpass

GH_API = "https://api.github.com"
GITHUB_REPO = f"connectivity-{COUNTRY_ISO3.lower()}"   # repository that will be created
GITHUB_OWNER = ""                                      # empty = your own account


def _gh(method, path, token, **kw):
    return requests.request(
        method, GH_API + path, timeout=90,
        headers={"Authorization": f"Bearer {token}",
                 "Accept": "application/vnd.github+json",
                 "X-GitHub-Api-Version": "2022-11-28"}, **kw)


def _as_token(value):
    """Coerce whatever a secret store returned into a clean token string.

    Colab's userdata.get() does not always hand back a plain string — depending on
    the runtime version it can return a mapping. Never assume; normalise.
    """
    if value is None:
        return ""
    if isinstance(value, dict):
        for k in ("value", "token", "secret", "GITHUB_TOKEN", "data"):
            if k in value:
                value = value[k]
                break
        else:
            value = next(iter(value.values()), "")
    if isinstance(value, (bytes, bytearray)):
        value = value.decode("utf-8", "ignore")
    return str(value).strip()


def _plausible(tok):
    """Cheap sanity check: a GitHub token is one long word, never a sentence."""
    return bool(tok) and len(tok) >= 20 and not any(c.isspace() for c in tok)


def _masked(tok):
    return f"{tok[:4]}{'•' * 8}{tok[-4:]}" if len(tok) > 12 else "•" * len(tok)


def ask_token():
    """Environment, then Colab secrets, then a hidden prompt. Never printed, never stored."""
    # 1 · environment variable
    tok = _as_token(os.environ.get("GITHUB_TOKEN"))
    if _plausible(tok):
        print(f"Token read from the GITHUB_TOKEN environment variable ({_masked(tok)}).")
        return tok

    # 2 · Colab Secrets panel
    if IN_COLAB:
        try:
            from google.colab import userdata
            tok = _as_token(userdata.get("GITHUB_TOKEN"))
            if _plausible(tok):
                print(f"Token read from the Colab Secrets panel ({_masked(tok)}).")
                return tok
            if tok:
                print("The Colab secret GITHUB_TOKEN does not look like a token — ignoring it.")
        except Exception as e:
            if "SecretNotFound" not in type(e).__name__:
                print(f"Colab Secrets unavailable ({type(e).__name__}) — falling back to the prompt.")

    # 3 · hidden prompt, with two attempts
    for attempt in (1, 2):
        tok = _as_token(getpass.getpass("GitHub token (hidden input — nothing is stored): "))
        if _plausible(tok):
            print(f"Token received ({_masked(tok)}).")
            return tok
        if attempt == 1:
            print("That does not look like a GitHub token (too short, or it contains a space). "
                  "Paste it again — it starts with ghp_ or github_pat_.")
    return ""


def publish_to_github(repo_name=None, owner=None, token=None, files=None,
                      branch="main", private=False, wait=True):
    """Create the public repository, upload the outputs, enable Pages, return the live URL."""
    repo_name = repo_name or GITHUB_REPO
    owner = owner or GITHUB_OWNER
    files = files or PUBLISH_FILES
    token = _as_token(token) or ask_token()
    if not token:
        raise RuntimeError("No token provided — publication cancelled.")

    # 1 · authenticate ------------------------------------------------------
    me = _gh("GET", "/user", token)
    if me.status_code != 200:
        raise RuntimeError(f"Authentication failed ({me.status_code}). Check the token, its "
                           "expiry date, and that the 'repo' scope is ticked.")
    login = me.json()["login"]
    owner = owner or login
    print(f"Authenticated as {login}" + (f" · publishing under {owner}" if owner != login else ""))

    # 2 · repository --------------------------------------------------------
    r = _gh("GET", f"/repos/{owner}/{repo_name}", token)
    if r.status_code == 200:
        branch = r.json().get("default_branch", branch)
        if r.json().get("private"):
            callout("This repository is <b>private</b>. GitHub Pages will not serve it publicly on "
                    "a free plan — make it public in Settings, or choose another name.", "risk")
        print(f"Repository {owner}/{repo_name} already exists — updating it (branch '{branch}').")
    else:
        body = {"name": repo_name, "private": private, "auto_init": True,
                "description": (f"{COUNTRY_NAME}: population-weighted connectivity indicators "
                                f"from Ookla Speedtest Open Data and WorldPop, "
                                f"{LATEST_Y} Q{LATEST_Q} — EN/FR"),
                "homepage": f"https://{owner}.github.io/{repo_name}/"}
        r = (_gh("POST", "/user/repos", token, json=body) if owner == login
             else _gh("POST", f"/orgs/{owner}/repos", token, json=body))
        if r.status_code not in (200, 201):
            raise RuntimeError(f"Could not create the repository ({r.status_code}): "
                               f"{r.json().get('message', r.text)}")
        branch = r.json().get("default_branch", branch)
        print(f"Repository created: {owner}/{repo_name} "
              f"({'private' if private else 'PUBLIC'}, default branch '{branch}')")
        time.sleep(2)

    # 3 · upload ------------------------------------------------------------
    print(f"\nUploading {len(files)} files …")
    uploaded = 0
    for f in files:
        if f.stat().st_size > 95e6:
            print(f"  SKIP {f.name} — above the 100 MB limit of the Contents API")
            continue
        sha = None
        g = _gh("GET", f"/repos/{owner}/{repo_name}/contents/{f.name}?ref={branch}", token)
        if g.status_code == 200 and isinstance(g.json(), dict):
            sha = g.json().get("sha")
        payload = {"message": f"{'Update' if sha else 'Add'} {f.name}",
                   "content": base64.b64encode(f.read_bytes()).decode(), "branch": branch}
        if sha:
            payload["sha"] = sha
        u = _gh("PUT", f"/repos/{owner}/{repo_name}/contents/{f.name}", token, json=payload)
        ok = u.status_code in (200, 201)
        uploaded += ok
        print(f"  {'OK  ' if ok else 'FAIL'} {f.name:<44} {f.stat().st_size/1024:8,.0f} KB"
              + ("" if ok else f"   -> {u.json().get('message', u.status_code)}"))
    print(f"{uploaded}/{len(files)} files uploaded.")

    # 4 · GitHub Pages ------------------------------------------------------
    print("\nEnabling GitHub Pages …")
    p = _gh("POST", f"/repos/{owner}/{repo_name}/pages", token,
            json={"source": {"branch": branch, "path": "/"}})
    if p.status_code in (201, 204):
        print("  Pages enabled.")
    elif p.status_code == 409:
        print("  Pages was already enabled — the site will rebuild automatically.")
    else:
        _gh("PUT", f"/repos/{owner}/{repo_name}/pages", token,
            json={"source": {"branch": branch, "path": "/"}})
        print(f"  Pages API answered {p.status_code}. If the URL 404s, switch it on manually: "
              "Settings → Pages → Deploy from a branch → main / root.")

    # 5 · wait for the first build ------------------------------------------
    site = f"https://{owner}.github.io/{repo_name}/"
    info = _gh("GET", f"/repos/{owner}/{repo_name}/pages", token)
    if info.status_code == 200:
        site = info.json().get("html_url", site)
    if wait:
        print("\nWaiting for the first build (up to 2 minutes) …")
        for _ in range(24):
            time.sleep(5)
            b = _gh("GET", f"/repos/{owner}/{repo_name}/pages/builds/latest", token)
            status = b.json().get("status") if b.status_code == 200 else None
            if status == "built":
                print("  Built and served.")
                break
            if status == "errored":
                print("  The build errored — check Settings → Pages in the repository.")
                break
        else:
            print("  Still building. The URL will answer shortly; reload if it 404s.")
    return site, f"https://github.com/{owner}/{repo_name}"


print("Ready. Run the next cell to publish.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Publish.  This cell asks for the token and does everything else by itself.
# ══════════════════════════════════════════════════════════════════════════════
try:
    SITE_URL, REPO_URL = publish_to_github()
    display(HTML(f"""
    <div style="font-family:{FONT};background:linear-gradient(120deg,{FOREST},{DEEP} 45%,{GREEN});
                color:#fff;border-radius:6px;padding:26px 30px;margin-top:14px">
      <div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:{GOLD}">
        PUBLISHED · PUBLICLY ACCESSIBLE · EN / FR</div>
      <div style="font-size:23px;font-weight:700;margin-top:8px">Your dashboard is online</div>
      <div style="margin-top:15px;font-size:15px">
        <a href="{SITE_URL}" target="_blank" style="color:#fff;background:rgba(255,255,255,.17);
           padding:10px 17px;border-radius:4px;text-decoration:none;font-weight:700">{SITE_URL}</a>
      </div>
      <div style="font-size:12.5px;color:#E6F6EE;margin-top:15px;line-height:1.6">
        Repository: <a href="{REPO_URL}" target="_blank" style="color:{GOLD}">{REPO_URL}</a><br>
        Anyone can open this link — no account, no login. Re-run this cell at any time to update
        the live site; the repository keeps every published version.
      </div>
      <div style="height:4px;background:{GOLD};margin-top:20px;width:110px"></div>
    </div>"""))
    print(f"\nPublic URL: {SITE_URL}")

except Exception as e:
    SITE_URL = None
    callout(f"<b>Publication did not complete.</b> {type(e).__name__}: {e}<br><br>"
            "Most frequent causes: the token has expired, the <code>repo</code> scope was not "
            "ticked, the repository name is already taken by another project, or this notebook is "
            "running without an interactive prompt (nbconvert, CI). In that last case, set the "
            "<code>GITHUB_TOKEN</code> environment variable before running. "
            "The manual route below always works.", "warn")

## If you would rather push it yourself

```bash
cd outputs
git init && git add -A && git commit -m "Connectivity dashboard — Ookla x WorldPop (EN/FR)"
git branch -M main
git remote add origin https://github.com/<your-account>/<your-repo>.git
git push -u origin main
```

Then, in the repository: **Settings → Pages → Source: *Deploy from a branch* → `main` / `/ (root)`
→ Save**. About a minute later the dashboard is live at
`https://<your-account>.github.io/<your-repo>/` — because the file was deliberately named
`index.html`, which is what a web server returns for the root of a site.

No command line at all? On github.com: **New repository** → tick **Public** → *uploading an existing
file* → drag the whole `outputs` folder in → *Commit changes* → then **Settings → Pages** as above.

## Checking that it really is public

Open the URL in a **private browsing window**, or send it to someone who is not signed in to
GitHub. If it loads there, and the EN / FR switch works, it is genuinely public. Three things break
this in practice:

- the repository is **private** — Pages then needs a paid plan and returns 404 on the free one;
- **Settings → Pages** was never saved, so no site was ever built;
- the entry file is not called `index.html`, so the root URL has nothing to serve.

## Two things to do before you push

1. **Check the licence with your legal service.** CC BY-NC-SA is unusual for a public statistical
   portal, and the non-commercial clause travels with every derivative.
2. **Read your own limitations statement aloud, in both languages.** If you would not be
   comfortable defending it in front of a journalist, it is not finished.

# 18 · Exercises

Do them in order; each one takes 5 to 15 minutes and each changes a figure you have just published.

**1 · Change the country.** Set `COUNTRY_ISO3` to a neighbour and run all. Compare the population
coverage of measurement. Why does it differ so much between two countries of similar size?

**2 · Break the weighting on purpose.** Replace `d_median_pop` with `d_median_tile` in the ranking
chart. Which districts move, and in which direction? Write one sentence explaining the movement to
a non-statistician.

**3 · Change the denominator.** The `pct_above_bb` indicator is computed over the *measured*
population. Recompute it over the *total* population, treating unmeasured people as unknown rather
than as zero — then as zero. You now have three legitimate numbers. Which one would you publish,
and what exactly would the footnote say?

**4 · Test the urbanisation thresholds.** Move `URBAN_DENS_MIN` from 1500 to 1000 and to 2500.
How much of the urban–rural gap is a property of the data, and how much of your threshold?

**5 · Add a quarter-on-quarter change map.** Compute the difference in population-weighted median
speed between the oldest and newest quarter for each administrative unit, and map it with a
diverging palette (brick → white → green). Careful: a unit that gained *tiles* may lose *speed*
purely by composition.

**6 · Go to 100 m.** Set `WORLDPOP_RES = "100m"`. The population cell is now smaller than the Ookla
tile, so the allocation rule inverts. Does the national population-weighted median move by more
than 1 Mbps? If not, you have just justified using the 1 km grid in production.

**7 · Confront it with official data.** Join your table to the ITU, operator or census figures you
already have for two or three regions. Do they rank in the same order? Rank correlation is a
sufficient test for an experimental indicator; level agreement is not required and should not be
expected.

**8 · Prepare the honest paragraph.** Write the three sentences you would say if a journalist asked
"so what is the average internet speed in my district?". They belong in your README, not in your
head.

# 19 · Sources

| Source | Reference | Licence |
|---|---|---|
| **Ookla® Speedtest Open Data** | `github.com/teamookla/ookla-open-data` — global performance tiles, quarterly since Q1 2019, Web-Mercator z16 | **CC BY-NC-SA 4.0** |
| **WorldPop** | `worldpop.org` — Global gridded population 2000–2020, UN-adjusted, University of Southampton | CC BY 4.0 |
| **geoBoundaries** | `geoboundaries.org` — gbOpen release, W. M. Geolab, William & Mary | CC BY 4.0 |
| **DuckDB** | `duckdb.org` — in-process analytical database, `httpfs` extension for remote parquet | MIT |
| **ITU** | *Measuring digital development: Facts and Figures* — reference thresholds for usable broadband | — |

**Attribution required in any publication derived from this notebook:**

> Contains information from Ookla® Speedtest Open Data, © Ookla, LLC, used under CC BY-NC-SA 4.0.
> Ookla trademarks are the property of Ookla, LLC. This product is not endorsed by or affiliated
> with Ookla. Population data © WorldPop (CC BY 4.0). Boundaries © geoBoundaries (CC BY 4.0).

---

<div style="background:linear-gradient(120deg,#00553A 0%,#00704A 45%,#00A86A 100%);
            padding:26px 30px;border-radius:6px;color:#fff;font-family:Calibri,sans-serif">
  <div style="font-size:10.5px;font-weight:700;letter-spacing:3px;color:#F5C242">END OF LABORATORY</div>
  <div style="font-size:24px;font-weight:700;margin-top:7px">You now have a public statistical product.</div>
  <div style="font-size:13.5px;color:#E6F6EE;margin-top:9px;line-height:1.6">
    A live public URL, a documented method, an explicit statement of limitations, and a licence
    that survives legal review. Re-run the notebook with another <code>COUNTRY_ISO3</code> and the
    whole product is rebuilt.
  </div>
  <div style="height:4px;background:#F5C242;margin-top:20px;width:110px"></div>
</div>